## 🤖 Semantic Model Data Agent Readiness Analyzer v2.2.2

> 💡 **Core Principle:** *"AI is only as good as the data behind it."* Before applying any Copilot or Data Agent feature, ensure your foundational semantic model is solid. If the data is messy or inconsistent, the AI's answers will be too.

> 📘 **Aligned with official guidance:** [semantic-model-ai-readiness.md](https://github.com/microsoft/skills-for-fabric/blob/main/skills/semantic-model-authoring/references/semantic-model-ai-readiness.md) | [Prepare your data for AI](https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai) | [Copilot Best Practices](https://learn.microsoft.com/power-bi/create-reports/copilot-semantic-models) | [Optimize your semantic model for Copilot](https://learn.microsoft.com/power-bi/create-reports/copilot-evaluate-data)

### What's New in v2.2.2 (Microsoft Reference-Aligned)

- 🆕 **Consumption Mode Framework** — Emphasizes "ask the user first": Reports only vs Conversational BI vs Both, determines required investment in model quality and configuration
- 🆕 **Core Principles highlighted** — Foundation first, Literal interpretation, Explicit over implicit, Iterate from observation
- 🆕 **Editing Capability Routing enhanced** — Clear guidance on which fixes require Power BI Desktop (TOM edits) vs PBIP files vs MCP vs cannot-edit items
- 🆕 **6-Section Readiness Checklist structure** — Aligned to Microsoft reference: (0) Gathering Business Context, (1) Model Architecture, (2) Naming, (3) Descriptions, (4) AI Instructions, (5) AI Data Schema, (6) Verified Answers
- 🆕 **Do/Don't guidance for each section** — Incorporates Microsoft reference best practices and anti-patterns
- 🆕 **Data Schema + Synonyms integration** — Clarifies AI Data Schema selection and synonym-via-AI-Instructions approach
- 🆕 **Verified Answers workflow guidance** — Cannot be automated; user configures in Desktop, agent suggests candidates
- 🆕 **Aligned with [Optimize your semantic model for Copilot](https://learn.microsoft.com/power-bi/create-reports/copilot-evaluate-data)** — adds Hierarchies check (1.17) and Copilot 200-character description window check (1.18)

### What's New in v2.2.1


### 📋 Prerequisites
- Run inside a **Microsoft Fabric** workspace notebook
- The semantic model must be published to a Fabric workspace
- You need **Build** (or higher) permissions on the semantic model
- `semantic-link-labs` and `httpx` will be installed automatically in the next cell
- Access to Power BI MCP Server for Prep for AI configuration validation
- **Recommended:** Run Best Practice Analyzer and Memory Analyzer before this notebook

---

## 🎯 Microsoft Reference Framework v2.2.2

### Consumption Mode Decision Matrix

**Ask the user first: How will this model be consumed?**

| **Consumption Mode** | **Description** | **Required Investment** |
|---|---|---|
| **📊 Reports Only** | Users access data via pre-built reports & dashboards | Foundation: architecture, naming, star schema |
| **💬 Conversational BI** | Copilot, Data Agents, or Q&A for ad-hoc questions | Full: foundation + AI metadata (descriptions, AI instructions, AI Data Schema) |
| **🔄 Both** | Mixed consumption (reports + conversational) | Full investment + Verified Answers for critical questions |

**Impact:** Determines scope of readiness review. Don't over-invest in Conversational BI features if the model is Reports-only.

---

### 🔑 Four Core Principles

These principles guide every decision in the readiness review:

1. **Foundation First** 
   - A weak model cannot be rescued by AI instructions or Verified Answers
   - Star schema, explicit measures, and correct data types are prerequisites
   - If section 1 (Model Architecture) fails, fixing section 4 (AI Instructions) will not help

2. **Literal Interpretation**
   - Copilot reads names and descriptions literally
   - `AMT` ≠ `Sales Amount` to a language model
   - Use business-friendly, human-readable terminology (Section 2: Naming)

3. **Explicit Over Implicit**
   - Implicit measures, report-scoped measures, and hidden business logic are invisible to Copilot
   - Everything reachable by natural language must live in the model as an explicit measure with metadata
   - Use explicit DAX measures instead of implicit aggregations (Section 1: Model Architecture)

4. **Iterate from Observation**
   - Don't guess. Write descriptions and AI instructions based on real user prompts and observed Copilot failures
   - Test with actual questions; refine based on results
   - Validate with domain experts and end-users before deployment

---

### 🔧 Editing Capability Routing (What Tool to Use?)

| **What to Edit** | **Where It Lives** | **How to Edit** | **Agent Can?** |
|---|---|---|---|
| **Names, descriptions, measures, data types, relationships** | TOM model metadata | Power BI Desktop | ✅ Yes (via MCP) |
| **Synonyms, AI Instructions, AI Data Schema** | PBIP files (.json) | PBIP project files or Fabric `updateDefinition` | ❌ Not directly |
| **Verified Answers** | PBIP + Report layer | Power BI Desktop + live testing | ❌ Suggest only |
| **Hidden flags, summarization, data categories** | TOM model metadata | Power BI Desktop | ✅ Yes (via MCP) |

**Rule:** When you cannot edit something automatically, refer the user to the appropriate tool.

---

In [ ]:
# Install required packages
%pip install semantic-link-labs httpx --quiet --upgrade

### ⚙️ Configuration — Set Your Parameters
Update the values below, then **Run all cells**.

In [ ]:
import sempy.fabric as fabric
import pandas as pd
import re
import warnings
import json
from datetime import datetime
warnings.filterwarnings('ignore')
from IPython.display import display, HTML, Markdown

# ============================================================
# 🔧  PARAMETERS — update these values
# ============================================================
dataset   = "ContosoSalesData"           # Name or ID of your semantic model
workspace = "YourWorkspaceName"          # Name or ID of your Fabric workspace

# Optional: Set to True if you've already configured Prep for AI
prep_for_ai_configured = False

# Optional: Set to True if this is a Direct Lake model
is_direct_lake = False
# ============================================================

print("=" * 70)
print("  🤖 SEMANTIC MODEL DATA AGENT READINESS ANALYZER v2.2.2")
print("=" * 70)
print(f"\n  Model     : {dataset}")
print(f"  Workspace : {workspace}")
print(f"  Timestamp : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n" + "=" * 70)
print("\nStarting analysis ...\n")

### 📦 Load Semantic Model Metadata

In [ ]:
print("Loading semantic model metadata ...")
print("-" * 70)

try:
    tables_df        = fabric.list_tables(dataset=dataset, workspace=workspace)
    columns_df       = fabric.list_columns(dataset=dataset, workspace=workspace)
    measures_df      = fabric.list_measures(dataset=dataset, workspace=workspace)
    relationships_df = fabric.list_relationships(dataset=dataset, workspace=workspace)
except Exception as e:
    print(f"❌ ERROR loading metadata: {e}")
    print("\nVerify:")
    print("  1. Dataset and workspace names are correct")
    print("  2. You have Build permissions on the semantic model")
    print("  3. The semantic model is published to Fabric")
    raise

# Helper functions
def is_hidden_mask(df, col='Is Hidden'):
    if col not in df.columns:
        return pd.Series([False] * len(df), index=df.index)
    return df[col].apply(lambda v: str(v).lower() == 'true' if not isinstance(v, bool) else v)

def missing_desc_mask(df, col='Description'):
    if col not in df.columns:
        return pd.Series([True] * len(df), index=df.index)
    return df[col].isna() | (df[col].astype(str).str.strip() == '')

# Filter system tables
SYS_PATTERN = r'^(DateTableTemplate_|LocalDateTable_)'
system_mask = tables_df['Name'].str.match(SYS_PATTERN, na=False)
tables_df   = tables_df[~system_mask].copy()

# Separate visible and hidden
hidden_tbl_mask = is_hidden_mask(tables_df)
hidden_col_mask = is_hidden_mask(columns_df)
hidden_msr_mask = is_hidden_mask(measures_df)

visible_tables   = tables_df[~hidden_tbl_mask]
visible_columns  = columns_df[~hidden_col_mask]
visible_measures = measures_df[~hidden_msr_mask]

# Remove row-number columns
if 'Is Row Number' in columns_df.columns:
    rn_mask = columns_df['Is Row Number'].apply(lambda v: str(v).lower() == 'true' if not isinstance(v, bool) else v)
    visible_columns = visible_columns[~rn_mask[visible_columns.index]]

print(f"✅ Metadata loaded for: '{dataset}'")
print()
print(f"  Visible tables   : {len(visible_tables):>4}  (total incl. hidden: {len(tables_df)})")
print(f"  Visible columns  : {len(visible_columns):>4}  (total incl. hidden: {len(columns_df)})")
print(f"  Visible measures : {len(visible_measures):>4}  (total incl. hidden: {len(measures_df)})")
print(f"  Relationships    : {len(relationships_df):>4}")
print()

# Global trackers
check_scores = {}  # {check_key: {'achieved': n, 'max': n, 'weight': n}}
all_issues   = []  # [(severity, check, description)]  # severity: critical | important | recommended

---
# 0️⃣ SCOPING & DESIGN ASSESSMENT

**Before diving into technical validation, ensure your Data Agent has a clear, narrow scope.**

> ⚠️ **Key Insight from Microsoft:** "Agents with laser-focused missions deliver value. General-purpose 'everything agents' fall short due to performance issues, confused logic, low trust, and maintenance challenges."

### Why Narrow Scope Matters

**Avoid:** ❌ General-purpose "everything agent" covering:
- Sales & Revenue Analytics
- Inventory & Supply Chain  
- Marketing & Campaign Performance
- Customer Support & Experience
- Finance & Forecasting
- Order Fulfillment & Logistics

**Prefer:** ✅ Domain-specific agents like:
- **Sales Intelligence Agent** — real-time order data, sales KPIs only
- **RevOps Intelligence Agent** — marketing campaigns + sales correlation only
- **Customer Experience Agent** — support tickets + satisfaction metrics only

### Scoping Checklist (Manual Assessment Required)

Before building your Data Agent, answer these questions:


---

## 📋 Six-Section Readiness Checklist (Microsoft Reference Structure)

**Complete sections in order — do not skip ahead.** If an earlier section fails, fixing later sections will not help.

| **Section** | **Focus** | **Do** | **Don't** |
|---|---|---|---|
| **0. Business Context** | Gather user personas, key questions, scope | Interview users; list top 10-15 questions | Assume scope; skip discovery |
| **1. Model Architecture** | Foundation quality: star schema, explicit measures, data types | Use star schema; create explicit DAX measures; correct data types | Leave flat models; rely on implicit aggregations |
| **2. Naming** | Business-friendly names for Copilot literal interpretation | Use full words (not abbreviations); avoid jargon | Use technical names, CamelCase, UPPER_CASE |
| **3. Descriptions** | Concise context (200-char limit for Copilot); front-load key info | Lead with purpose; include units/ranges; describe calculations | Generic auto-generated descriptions; bury key info |
| **4. AI Instructions** | Metric routing, terminology, time definitions, clarification rules | Define routing for ambiguous terms; list business abbreviations | Duplicate descriptions; encode should-be-measures as instructions |
| **5. AI Data Schema** | Control Copilot exposure: include business tables, exclude helpers | Include only business-relevant columns; exclude PKs/FKs/GUIDs; add synonyms | Expose everything; dilute signal Copilot uses |
| **6. Verified Answers** | Pre-built trusted responses for top questions | Configure top 3-5 most-asked questions in Desktop; validate with users | Attempt to auto-generate; skip user validation |

**Impact:** Sections 0-5 can be partially automated via MCP. Section 6 requires manual Desktop configuration. Skipping = models that disappoint at scale.

---

In [ ]:
print("=" * 70)
print("CHECK 0: SCOPING & DESIGN ASSESSMENT (MANUAL)")
print("=" * 70)
print("\n🎯 CRITICAL: Complete this scoping exercise BEFORE technical configuration.\n")
print("✅ SCOPING CHECKLIST:")
print()
print("   📝 1. WHO ARE THE PRIMARY USERS?")
print("      Define specific user personas (e.g., 'sales analysts', 'regional managers')")
print("      NOT generic 'business users' or 'everyone in the company'")
print()
print("   📝 2. WHAT ARE THE MAIN FUNCTIONS? (List 3-5 specific tasks)")
print("      Examples:")
print("        ✅ 'Monitor real-time order volume by region'")
print("        ✅ 'Compare campaign ROI across marketing channels'")
print("        ✅ 'Identify top complaint categories by geography'")
print("      NOT:")
print("        ❌ 'Answer any business question'")
print("        ❌ 'Provide insights from all data'")
print()
print("   📝 3. TOP 10-15 QUESTIONS USERS WILL ASK")
print("      List actual expected questions:")
print("        • 'What are our top-selling products this week?'")
print("        • 'Which warehouses are running low on stock?'")
print("        • 'What's the ROI of Q1 influencer campaigns in APAC?'")
print()
print("   📝 4. WHAT'S OUT OF SCOPE?")
print("      Clearly define what this agent WON'T handle:")
print("        • HR/employee queries → not included")
print("        • Financial forecasting → separate agent")
print("        • Unstructured documents → not in scope")
print()
print("   📝 5. DATA AVAILABILITY CHECK")
print("      For each key question, verify:")
print("        ☐ Data exists in Fabric (Lakehouse/Warehouse/Semantic Model)")
print("        ☐ Data is current (refresh frequency meets requirements)")
print("        ☐ Data quality is acceptable (no major gaps/errors)")
print("        ☐ Questions are answerable with existing data")
print()
print("   📝 6. SECURITY REQUIREMENTS")
print("      Define who can access this Data Agent:")
print("        ☐ Identify user personas who will query this agent")
print("        ☐ Document Row Level Security (RLS) requirements")
print("        ☐ List sensitive data fields to exclude from AI Data Schema")
print("        ☐ Confirm compliance with data privacy policies (GDPR, HIPAA, etc.)")
print("        ☐ Define if agent should be internal-only or externally accessible")
print("        ☐ Verify workspace-level permissions are appropriately scoped")
print()
print("⚠️  WARNING SIGNS OF SCOPE PROBLEMS:")
print("   • More than 20 tables selected in AI Data Schema")
print("   • Agent serves multiple unrelated business domains")
print("   • Users from 5+ different departments")
print("   • Mix of real-time and historical analytics")
print("   • Questions require 'why' or open-ended analysis")
print()
print("💡 DESIGN PATTERNS:")
print()
print("   ┌─────────────────────────────────────────────────────────┐")
print("   │ PATTERN 1: Domain-Specific Agents (RECOMMENDED)        │")
print("   ├─────────────────────────────────────────────────────────┤")
print("   │ • Sales Intelligence Agent → Eventhouse + Sales Model  │")
print("   │ • RevOps Agent → Marketing Data + Sales KPIs           │")
print("   │ • Cx Intelligence Agent → Support Lakehouse only        │")
print("   │                                                         │")
print("   │ Benefits: High accuracy, clear ownership, maintainable │")
print("   └─────────────────────────────────────────────────────────┘")
print()
print("   ┌─────────────────────────────────────────────────────────┐")
print("   │ PATTERN 2: Multi-Agent Orchestration (ADVANCED)        │")
print("   ├─────────────────────────────────────────────────────────┤")
print("   │ Main Orchestrator Agent routes to:                     │")
print("   │   → Sales Intelligence Agent                            │")
print("   │   → Marketing Intelligence Agent                        │")
print("   │   → Customer Support Agent                              │")
print("   │   → Finance Agent                                       │")
print("   │                                                         │")
print("   │ Use when: Users need cross-domain questions            │")
print("   └─────────────────────────────────────────────────────────┘")
print()
print()

# Score based on awareness
score = 10  if prep_for_ai_configured else 0
check_scores['0_scoping'] = {'achieved': score, 'max': 10, 'weight': 2}
if not prep_for_ai_configured:
    all_issues.append(("critical", "0 Scoping", "Scoping questionnaire not completed"))
    
print(f"📊 Score: {score}/10 (manual assessment required)")
print()
print("=" * 70)

---
# 1️⃣ SEMANTIC MODEL OPTIMIZATION

This section validates foundational model quality that directly impacts Data Agent accuracy and performance.

## 1.1 ⭐ Star Schema Validation

**Why it matters:** Data Agent's DAX generation relies on clear fact-dimension relationships. Flat/denormalized models produce unreliable results.

**What's checked:**
- No relationships (flat model) — **CRITICAL**
- Many-to-many relationships — **should be rare**
- Bidirectional cross-filtering — **use sparingly**
- Isolated visible tables — **connect or hide**

In [ ]:
print("=" * 70)
print("CHECK 1.1: STAR SCHEMA VALIDATION")
print("=" * 70)

issues = []
score = 20  # Max points

if relationships_df.empty:
    print("❌ CRITICAL: No relationships found — flat/denormalized model detected.")
    print("\n   This is a BLOCKING issue for Data Agent accuracy.")
    print("   Refactor into a star schema with clear fact and dimension tables.")
    print("\n   📚 Learn: https://learn.microsoft.com/power-bi/guidance/star-schema")
    issues.append(("critical", "1.1 Star Schema", "No relationships found — flat model"))
    score = 0
else:
    # Many-to-many relationships
    m2m_col = next((c for c in ['Multiplicity', 'Cardinality'] if c in relationships_df.columns), None)
    if m2m_col:
        m2m = relationships_df[
            relationships_df[m2m_col].astype(str).str.contains('ManyToMany|Many.*Many', case=False, na=False)
        ]
        if not m2m.empty:
            print(f"⚠️  {len(m2m)} many-to-many relationship(s) found:")
            for _, r in m2m.iterrows():
                ft = r.get('From Table', 'Unknown')
                tt = r.get('To Table', 'Unknown')
                print(f"     • {ft}  ↔  {tt}")
            print("\n   M:M relationships reduce DAX accuracy and performance.")
            print("   ACTION: Introduce bridge tables to resolve these.")
            issues.append(("important", "1.1 Star Schema", f"{len(m2m)} many-to-many relationship(s)"))
            score -= min(6, len(m2m) * 3)

    # Bidirectional cross-filter
    cf_col = next((c for c in ['Cross Filter Direction', 'Cross Filtering Behavior'] if c in relationships_df.columns), None)
    if cf_col:
        bidir = relationships_df[
            relationships_df[cf_col].astype(str).str.contains('Both', case=False, na=False)
        ]
        if not bidir.empty:
            print(f"⚠️  {len(bidir)} bidirectional relationship(s):")
            for _, r in bidir.head(5).iterrows():
                print(f"     • {r.get('From Table', '?')} ↔ {r.get('To Table', '?')}")
            print("\n   Bidirectional filtering can create ambiguity in DAX generation.")
            print("   ACTION: Review if these are necessary; consider alternatives.")
            issues.append(("important", "1.1 Star Schema", f"{len(bidir)} bidirectional relationship(s)"))
            score -= min(4, len(bidir) * 2)

    # Isolated visible tables
    related_tables = set()
    if 'From Table' in relationships_df.columns:
        related_tables = set(relationships_df['From Table'].dropna()) | set(relationships_df['To Table'].dropna())
    visible_tbl_names = set(visible_tables['Name'].dropna())
    isolated = visible_tbl_names - related_tables
    # Exclude parameter tables
    isolated = {t for t in isolated if not any(x in t.lower() for x in ['parameter', ' parameter', '_param'])}
    
    if isolated:
        print(f"\n⚠️  {len(isolated)} visible table(s) with no relationships:")
        for t in sorted(isolated)[:8]:
            print(f"     • {t}")
        if len(isolated) > 8:
            print(f"     ... and {len(isolated) - 8} more")
        print("\n   ACTION: Connect these tables to the model or hide them.")
        issues.append(("important", "1.1 Star Schema", f"{len(isolated)} isolated visible table(s)"))
        score -= min(6, len(isolated) * 2)

    if not issues:
        print("✅ PASSED: Relationship structure follows star schema design.")

score = max(0, score)
print(f"\n📊 Score: {score}/20")
check_scores['1.1_star_schema'] = {'achieved': score, 'max': 20, 'weight': 3}
all_issues.extend(issues)

## 1.2 ⚡ Best Practice Analyzer

**Why it matters:** BPA identifies 60+ performance, DAX quality, and error prevention issues that affect Data Agent response time and reliability.

**What's checked:** All BPA rules (performance, DAX, formatting, maintenance)

In [ ]:
print("=" * 70)
print("CHECK 1.2: BEST PRACTICE ANALYZER")
print("=" * 70)

issues = []
score = 15  # Max points

try:
    import sempy_labs as labs
    bpa_results = labs.run_model_bpa(dataset=dataset, workspace=workspace)

    if bpa_results is not None and not bpa_results.empty:
        sev_col = 'Severity' if 'Severity' in bpa_results.columns else None
        print(f"⚠️  {len(bpa_results)} BPA recommendation(s) found:\n")
        
        if sev_col:
            sev_counts = bpa_results[sev_col].value_counts()
            for sev in ['Error', 'Warning', 'Info']:
                if sev in sev_counts.index:
                    print(f"   {sev}: {sev_counts[sev]}")
        
        # Score based on severity
        if   len(bpa_results) >= 30: score = 3
        elif len(bpa_results) >= 20: score = 6
        elif len(bpa_results) >= 10: score = 9
        elif len(bpa_results) >= 5:  score = 12
        else:                        score = 14

        print("\nTop issues (click to expand for full details):")
        display(bpa_results.head(20))
        
        issues.append(("important", "1.2 BPA", f"{len(bpa_results)} recommendations from Best Practice Analyzer"))
        
        print("\n📚 Learn more: https://learn.microsoft.com/power-bi/transform-model/service-notebooks")
    else:
        print("✅ PASSED: No BPA issues found!")

except ImportError:
    print("ℹ️  Semantic Link Labs not available for BPA.")
    print("   Install: %pip install semantic-link-labs --upgrade")
    score = 10
except Exception as e:
    print(f"⚠️  Could not run BPA: {e}")
    print("\n   Run manually: fabric.run_model_bpa(dataset=dataset, workspace=workspace)")
    score = 10

print(f"\n📊 Score: {score}/15")
check_scores['1.2_bpa'] = {'achieved': score, 'max': 15, 'weight': 3}
all_issues.extend(issues)

## 1.3 💾 Memory Analyzer

**Why it matters:** Large model size impacts Data Agent performance and workspace capacity. Optimizing memory usage improves response times.

**What's checked:** Runs `fabric.model_memory_analyzer()` to analyze model memory usage and identify optimization opportunities

**Key insights:**
- Memory footprint by table and column
- High-cardinality columns that may need optimization
- Unused or redundant columns
- Data type optimization opportunities

In [ ]:
print("=" * 70)
print("CHECK 1.3: MEMORY ANALYZER")
print("=" * 70)

print("Running Model Memory Analyzer...\n")

try:
    # Run the model memory analyzer
    memory_results = fabric.model_memory_analyzer(dataset=dataset, workspace=workspace)
    
    print("✅ Memory Analyzer completed successfully!")
    print("\n📊 Results displayed above.")
    print("\n✅ Key actions from Memory Analyzer:")
    print("   • Remove unused columns")
    print("   • Optimize high-cardinality columns")
    print("   • Review column data types")
    print("   • Consider aggregations for large fact tables")
    
    if is_direct_lake:
        print("\n💡 DIRECT LAKE SPECIFIC:")
        print("   • Run V-Order optimization on Delta tables")
        print("   • Review Direct Lake fallback scenarios")
        print("   • Monitor Direct Lake mode in Query Performance Analyzer")
        print("\n   📚 https://learn.microsoft.com/fabric/fundamentals/direct-lake-understand-storage")
    
    # Full credit for running the analyzer
    score = 5
    issues = []

except Exception as e:
    print(f"⚠️  Could not run Memory Analyzer: {e}")
    print("\n📚 Documentation:")
    print("   https://learn.microsoft.com/power-bi/transform-model/service-notebooks#model-memory-analyzer")
    print("\n🔧 Manual alternative:")
    print("   Run in a separate cell: fabric.model_memory_analyzer(dataset=dataset, workspace=workspace)")
    
    # Partial credit
    score = 3
    issues = [("important", "1.3 Memory", "Memory Analyzer could not be executed")]

check_scores['1.3_memory'] = {'achieved': score, 'max': 5, 'weight': 1}
all_issues.extend(issues)
print(f"\n📊 Score: {score}/5")

## 1.4 🔍 Data Type Validation

**Why it matters:** Incorrect data types cause DAX errors, incorrect aggregations, and poor performance.

**What's checked:**
- Date columns stored as text
- Numeric columns stored as text
- Text columns with numeric names

In [ ]:
print("=" * 70)
print("CHECK 1.4: DATA TYPE VALIDATION")
print("=" * 70)

issues = []
score = 10

# Detect potential date columns stored as text
date_pattern = r'(date|dt_|dt$|_dt|day|month|year|quarter|fiscal|calendar|timestamp|time)'
text_date_cols = visible_columns[
    (visible_columns['Data Type'].astype(str).str.contains('string|text', case=False, na=False)) &
    (visible_columns['Column Name'].astype(str).str.lower().str.contains(date_pattern, na=False))
]

numeric_pattern = r'(amount|amt|quantity|qty|count|cnt|price|cost|rate|percent|pct|total|sum|value|val|number|num|nbr|id|key)$'
text_numeric_cols = visible_columns[
    (visible_columns['Data Type'].astype(str).str.contains('string|text', case=False, na=False)) &
    (visible_columns['Column Name'].astype(str).str.lower().str.contains(numeric_pattern, na=False))
]

if not text_date_cols.empty:
    print(f"⚠️  {len(text_date_cols)} potential date column(s) stored as text:")
    for _, row in text_date_cols.head(10).iterrows():
        print(f"     • {row.get('Table Name')}[{row.get('Column Name')}]")
    if len(text_date_cols) > 10:
        print(f"     ... and {len(text_date_cols) - 10} more")
    print("\n   ACTION: Convert to Date or DateTime type in Power Query.")
    issues.append(("important", "1.4 Data Types", f"{len(text_date_cols)} date-like columns stored as text"))
    score -= min(5, len(text_date_cols))

if not text_numeric_cols.empty:
    print(f"\n⚠️  {len(text_numeric_cols)} potential numeric column(s) stored as text:")
    for _, row in text_numeric_cols.head(10).iterrows():
        print(f"     • {row.get('Table Name')}[{row.get('Column Name')}]")
    if len(text_numeric_cols) > 10:
        print(f"     ... and {len(text_numeric_cols) - 10} more")
    print("\n   ACTION: Verify these are truly text; convert if numeric.")
    issues.append(("recommended", "1.4 Data Types", f"{len(text_numeric_cols)} numeric-like columns stored as text"))
    score -= min(3, len(text_numeric_cols))

if not issues:
    print("✅ PASSED: No obvious data type issues detected.")

score = max(0, score)
print(f"\n📊 Score: {score}/10")
check_scores['1.4_data_types'] = {'achieved': score, 'max': 10, 'weight': 2}
all_issues.extend(issues)

## 1.5 🏷️ Business-Friendly Naming

**Why it matters:** Data Agent interprets object names literally. Technical names like `DIM_CUST_01` or `F_SLS_AMT` provide no context.

**What's checked:** Database-style prefixes/suffixes, abbreviations, ALL_CAPS, cryptic names

**📐 Naming convention guidelines** (applied to tables, columns, and measures):

- **Avoid abbreviations and acronyms** — instead of `Std. Margin` or `S.M.`, write `Standard Margin`.
- **Avoid excessive numbers and grammar, and don't use Unicode characters** — keep names plain ASCII.
- **Avoid technical prefixes** like `DIM_` or `FACT_`.
- **Use standard casing with spaces** instead of `CamelCase` or `snake_case`.
- **Use special prefixes for technical fields or properties** to sort or identify them.
- **Do specify aggregations, units, and periods — but use a consistent syntax.** Choose one convention for each of the following:
  - **Periods** (previous year, previous month, yesterday, etc.) — instead of `Sales Last Year`, write `Sales (1YP)` or `Sales (PY)`. `Sales (1YP)` is preferable because it's easy to extend to `Sales (2YP)`.
  - **Aggregations** (MTD, YTD, Weekly Average, etc.) — instead of `Sales Month To Date PY`, write `Sales MTD (1YP)`.
  - **Units** (currency €, percent %, units (qty), lines, pieces (pcs), etc.) — layering units can get complex, e.g. `Sales (€) MTD (1YP)` alongside `Sales (Units) YTD (2YP)`.
  - **Comparisons** to targets, periods, or aggregates (e.g. `vs Budget`, `vs 1YP`, `vs Average`).

In [ ]:
import re
import pandas as pd

# -------------------------------------------------------------------------
# CONFIGURATION: PATTERNS FOR COPILOT READINESS
# -------------------------------------------------------------------------

## 1.6 📝 Descriptions Coverage

**Why it matters:** Descriptions are CRITICAL for Data Agent accuracy. Every object in your AI Data Schema MUST have a clear, concise description.

**What's checked:** Description coverage for visible tables, columns, and measures

In [ ]:
# =========================================================================
# CHECK 1.6: DESCRIPTIONS COVERAGE (ORIGINAL CODE)
# =========================================================================
print("=" * 70)
print("CHECK 1.6: DESCRIPTIONS COVERAGE")
print("=" * 70)

issues = []
score = 20  # Highest weight — most critical

tables_no_desc   = visible_tables[missing_desc_mask(visible_tables)]
cols_no_desc     = visible_columns[missing_desc_mask(visible_columns)]
measures_no_desc = visible_measures[missing_desc_mask(visible_measures)]

tbl_cov = 1 - len(tables_no_desc)   / max(len(visible_tables),   1)
col_cov = 1 - len(cols_no_desc)     / max(len(visible_columns),  1)
msr_cov = 1 - len(measures_no_desc) / max(len(visible_measures), 1)

print("Description Coverage:\n")
print(f"  Tables   : {len(visible_tables) - len(tables_no_desc):>4}/{len(visible_tables):<4} ({tbl_cov*100:>5.1f}%)")
print(f"  Columns  : {len(visible_columns) - len(cols_no_desc):>4}/{len(visible_columns):<4} ({col_cov*100:>5.1f}%)")
print(f"  Measures : {len(visible_measures) - len(measures_no_desc):>4}/{len(visible_measures):<4} ({msr_cov*100:>5.1f}%)")
print()

total_missing = len(tables_no_desc) + len(cols_no_desc) + len(measures_no_desc)

if total_missing == 0:
    print("✅ PASSED: All visible objects have descriptions.")
else:
    overall_cov = 1 - total_missing / max(len(visible_tables) + len(visible_columns) + len(visible_measures), 1)
    score = max(0, int(20 * overall_cov))

    if not tables_no_desc.empty:
        print(f"❌ {len(tables_no_desc)} table(s) missing descriptions:")
        for name in tables_no_desc['Name'].tolist()[:8]:
            print(f"     • {name}")
        if len(tables_no_desc) > 8:
            print(f"     ... and {len(tables_no_desc) - 8} more")

    if not cols_no_desc.empty:
        print(f"\n⚠️  {len(cols_no_desc)} column(s) missing descriptions (first 15):")
        for _, row in cols_no_desc.head(15).iterrows():
            print(f"     • {row.get('Table Name', '?')}[{row.get('Column Name', '?')}]")
        if len(cols_no_desc) > 15:
            print(f"     ... and {len(cols_no_desc) - 15} more")

    if not measures_no_desc.empty:
        print(f"\n⚠️  {len(measures_no_desc)} measure(s) missing descriptions (first 15):")
        for _, row in measures_no_desc.head(15).iterrows():
            print(f"     • {row.get('Table Name', '?')}[{row.get('Measure Name', '?')}]")
        if len(measures_no_desc) > 15:
            print(f"     ... and {len(measures_no_desc) - 15} more")

    print("\n🚨 CRITICAL: Add descriptions to ALL objects in your AI Data Schema before deployment.")
    issues.append(("critical", "1.6 Coverage", f"{total_missing} visible objects missing descriptions"))

print(f"\n📊 Coverage Score: {score}/20")
check_scores['1.6_descriptions'] = {'achieved': score, 'max': 20, 'weight': 3}


# =========================================================================
# CHECK 1.6.1: DESCRIPTIONS QUALITY (INFORMATIONAL AUDIT)
# =========================================================================
print("\n" + "=" * 70)
print("CHECK 1.6.1: DESCRIPTIONS QUALITY AUDIT")
print("=" * 70)

quality_issues = []

# First check: Are there ANY descriptions at all?
total_with_descriptions = (len(visible_tables) - len(tables_no_desc)) + \
                          (len(visible_columns) - len(cols_no_desc)) + \
                          (len(visible_measures) - len(measures_no_desc))

if total_with_descriptions == 0:
    print("❌ FAILED: No descriptions found in the model.")
    print("\n   🚨 CRITICAL: All visible objects must have descriptions for Data Agent to function properly.")
    print("   ACTION: Add descriptions to tables, columns, and measures before proceeding with quality checks.")
else:
    # Filter for objects that ALREADY have descriptions to audit their content
    tables_with_desc   = visible_tables[~missing_desc_mask(visible_tables)]
    cols_with_desc     = visible_columns[~missing_desc_mask(visible_columns)]
    measures_with_desc = visible_measures[~missing_desc_mask(visible_measures)]

    print(f"✅ Found {total_with_descriptions} descriptions to audit for quality.\n")

    def check_quality(name, desc):
        n = str(name).lower().strip()
        d = str(desc).lower().strip()
        
        if d == n: 
            return "Redundant: Matches name exactly"
        if len(d) < 15: 
            return "Too brief: Less than 15 characters"
        if any(p in d for p in ['tbd', 'todo', 'placeholder', 'n/a', 'test']): 
            return "Placeholder text detected"
        if d.startswith("this column") or d.startswith("this measure"): 
            return "Circular: Starts with 'This column/measure...'"
        return None

    # Audit Tables Quality
    table_quality_issues = []
    for _, row in tables_with_desc.iterrows():
        issue = check_quality(row['Name'], row['Description'])
        if issue: 
            table_quality_issues.append(f"Table: {row['Name']} → {issue}")
            quality_issues.append(f"Table: {row['Name']} → {issue}")

    # Audit Columns Quality
    column_quality_issues = []
    for _, row in cols_with_desc.iterrows():
        issue = check_quality(row['Column Name'], row['Description'])
        if issue: 
            column_quality_issues.append(f"Column: {row['Table Name']}[{row['Column Name']}] → {issue}")
            quality_issues.append(f"Column: {row['Table Name']}[{row['Column Name']}] → {issue}")

    # Audit Measures Quality
    measure_quality_issues = []
    for _, row in measures_with_desc.iterrows():
        issue = check_quality(row['Measure Name'], row['Description'])
        if issue: 
            measure_quality_issues.append(f"Measure: {row['Table Name']}[{row['Measure Name']}] → {issue}")
            quality_issues.append(f"Measure: {row['Table Name']}[{row['Measure Name']}] → {issue}")

    # Report quality audit results
    if not quality_issues:
        print("✅ PASSED: All descriptions meet AI quality standards.")
        print(f"\n   • Tables: {len(tables_with_desc)} checked ✓")
        print(f"   • Columns: {len(cols_with_desc)} checked ✓")
        print(f"   • Measures: {len(measures_with_desc)} checked ✓")
    else:
        print(f"ℹ️  QUALITY AUDIT RESULTS: {len(quality_issues)} descriptions could be improved:\n")
        
        if table_quality_issues:
            print(f"   📊 Tables ({len(table_quality_issues)} issues):")
            for item in table_quality_issues[:5]:
                print(f"      • {item}")
        
        if column_quality_issues:
            print(f"\n   📋 Columns ({len(column_quality_issues)} issues):")
            for item in column_quality_issues[:5]:
                print(f"      • {item}")
        
        if measure_quality_issues:
            print(f"\n   📐 Measures ({len(measure_quality_issues)} issues):")
            for item in measure_quality_issues[:5]:
                print(f"      • {item}")
        
        remaining = max(0, len(quality_issues) - 15)
        if remaining > 0:
            print(f"\n   ... and {remaining} more issues")

        print("\n💡 AI TIP: Use descriptions to provide business context that the object name doesn't explain.")
        print("   Avoid: 'Total Sales' → 'This measure calculates total sales'")
        print("   Better: 'Total Sales' → 'Sum of all invoiced sales amounts in USD, excluding returns and discounts'")

# Final merge of issues found in 1.6 (1.6.1 is informational, so it doesn't append to all_issues)
all_issues.extend(issues)

## 1.7 🔤 Synonyms / Linguistic Schema

**Why it matters:** Synonyms help Data Agent match natural language variations (revenue/income/sales → Total Revenue).

**What's checked:** Synonym configuration on visible objects

In [ ]:
print("=" * 70)
print("CHECK 1.7: SYNONYMS / LINGUISTIC SCHEMA")
print("=" * 70)

issues = []
score = 5

try:
    import sempy_labs as labs

    # Use Semantic Link Labs to list all synonyms
    df_synonyms = labs.list_synonyms(dataset=dataset, workspace=workspace)

    if df_synonyms is not None and not df_synonyms.empty:
        # Count synonyms by object type
        syn_by_type = df_synonyms.groupby('Object Type').size()

        print("✅ Synonyms configured:\n")
        for obj_type, count in syn_by_type.items():
            print(f"   {obj_type:<10} : {count:>4} object(s)")

        print(f"\n   Total: {len(df_synonyms)} synonym(s) across "
              f"{len(df_synonyms['Object Name'].unique())} object(s)")

        # ----------------------------------------------------------------
        # OVER-SYNONYMIZATION DETECTION
        # Flags identical synonym terms mapped to multiple distinct objects.
        # Per best practice: don't map the same synonym (e.g., 'Sales') to
        # both 'Order Count' and 'Total Revenue' — this confuses the AI.
        # ----------------------------------------------------------------
        syn_col = next((c for c in ['Synonym', 'Synonyms', 'Term'] if c in df_synonyms.columns), None)
        if syn_col:
            exploded = df_synonyms.copy()
            exploded[syn_col] = exploded[syn_col].astype(str).str.split(r'[,;|]')
            exploded = exploded.explode(syn_col)
            exploded[syn_col] = exploded[syn_col].str.strip().str.lower()
            exploded = exploded[exploded[syn_col] != '']

            dup_terms = (
                exploded.groupby(syn_col)['Object Name']
                .nunique()
                .reset_index(name='Object Count')
            )
            dup_terms = dup_terms[dup_terms['Object Count'] > 1]

            if not dup_terms.empty:
                print(f"\n⚠️  OVER-SYNONYMIZATION: {len(dup_terms)} term(s) mapped to multiple objects:")
                for _, r in dup_terms.head(8).iterrows():
                    targets = exploded[exploded[syn_col] == r[syn_col]]['Object Name'].unique()
                    print(f"     • '{r[syn_col]}'  →  {', '.join(targets[:5])}")
                print("\n   ❌ Don't over-synonymize: a single term mapped to multiple objects")
                print("      (e.g., 'Sales' → Order Count AND Total Revenue) confuses the AI.")
                issues.append(("important", "1.7 Synonyms",
                               f"{len(dup_terms)} synonym term(s) mapped to multiple objects"))
                score = max(2, score - 2)

        # Auto-generated synonym warning
        if 'State' in df_synonyms.columns:
            auto_gen = df_synonyms[
                df_synonyms['State'].astype(str).str.contains('Auto|Generated', case=False, na=False)
            ]
            if not auto_gen.empty:
                print(f"\nℹ️  {len(auto_gen)} auto-generated synonym(s) detected.")
                print("   Manually review and approve — auto-generated synonyms can be wrong")
                print("   (e.g., Copilot may guess 'Client' = 'Customer'; verify explicitly).")

        # Check coverage of visible measures
        if not measures_df.empty:
            measures_with_synonyms = df_synonyms[
                df_synonyms['Object Type'] == 'Measure'
            ]['Object Name'].unique()
            vis_msr_names = visible_measures['Measure Name'].tolist()
            msrs_without = [m for m in vis_msr_names if m not in measures_with_synonyms]

            if msrs_without:
                pct_covered = 1 - len(msrs_without) / max(len(vis_msr_names), 1)
                score = max(2, int(5 * pct_covered))
                print(f"\n⚠️  {len(msrs_without)} visible measure(s) without synonyms (showing first 10):")
                for m in msrs_without[:10]:
                    print(f"     • {m}")
                if len(msrs_without) > 10:
                    print(f"     ... and {len(msrs_without) - 10} more")
                issues.append(("recommended", "1.7 Synonyms",
                               f"{len(msrs_without)} measures without synonyms"))
            else:
                print("\n✅ All visible measures have synonyms!")

        print("\n📊 Synonym details:")
        display(df_synonyms.head(20))
    else:
        print("⚠️  No synonyms configured on any object.")
        print("\n   ACTION: Define synonyms for key measures and dimensions.")
        print("   PREFERRED METHOD: Use 'AI Instructions' (Prep for AI) in natural language,")
        print("                     e.g., \"VFR means Volume of Fruit Revenue\".")
        score = 1
        issues.append(("recommended", "1.7 Synonyms", "No synonyms configured"))

    print("\n💡 SYNONYM BEST PRACTICES:")
    print("   ✅ DO use AI Instructions to define synonyms in natural language")
    print("       (modern, version-controlled, optimized for the LLM)")
    print("   ✅ DO manually approve synonyms — auto-generated ones can be wrong")
    print("   ❌ DON'T use the legacy Q&A Setup dialog for new definitions")
    print("   ❌ DON'T map the same synonym to multiple distinct objects")
    print("       (e.g., 'Sales' → Order Count AND Total Revenue → AI confusion)")
    print("   ❌ DON'T expose cryptic acronyms (e.g., 'VFR') without defining them")
    print("       in AI Instructions — the LLM has no way to know your jargon.")
    print("\n   Synonym examples (use sparingly, one canonical target each):")
    print("   • Total Revenue → income, turnover, proceeds")
    print("   • Customer → client, account, buyer")

    print("\n📚 Learn more:")
    print("   https://learn.microsoft.com/power-bi/create-reports/copilot-semantic-models#linguistic-schema")
    print("   https://learn.microsoft.com/power-bi/natural-language/q-and-a-best-practices")

except Exception as e:
    print(f"ℹ️  Could not retrieve synonyms: {e}")
    print("\n   Manual check: Power BI Desktop → select object → Properties pane → Synonyms")
    print("   PREFERRED: Define synonyms in 'AI Instructions' (Prep for AI) instead of legacy Q&A Setup.")
    score = 3

print("\n🔔 PREREQUISITES (per Microsoft Learn):")
print("   • Q&A must be enabled to add a linguistic schema:")
print("     Power BI Desktop → File → Options → Current file → Data Load →")
print("     'Turn on Q&A to ask natural language questions about your data'")
print("   • To EXCLUDE a technical / redundant field from Copilot:")
print("     Q&A Setup → Synonyms → toggle OFF 'Include in Q&A' for that field.")
print("     This is the canonical way to hide a field from Copilot's linguistic context")
print("     when you can't simply hide it in the model.")
print("   • You can use Copilot itself in Q&A Setup to SUGGEST synonyms,")
print("     then curate them manually — do NOT accept auto-generated synonyms blindly.")

print(f"\n📊 Score: {score}/5")
check_scores['1.7_synonyms'] = {'achieved': score, 'max': 5, 'weight': 1}
all_issues.extend(issues)


## 1.8 🏷️ Row Labels

**Why it matters:** Row labels define the primary descriptor for dimension tables, helping Data Agent identify which column to display (e.g., "Customer Name" instead of "Customer ID").

**What's checked:** Row label configuration on dimension tables

In [ ]:
print("=" * 70)
print("CHECK 1.8: ROW LABELS FOR DIMENSION TABLES")
print("=" * 70)

issues = []
score = 5

try:
    # Using Semantic Link Labs TOM (Tabular Object Model) access
    # Row labels are exposed through Table.RowLabel TOM property
    # This property is fully represented in TMDL and accessible via XMLA
    from sempy_labs.tom import connect_semantic_model

    tables_with_row_labels = []
    tables_without = []
    all_user_tables = []

    with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
        for table in tom.model.Tables:
            # Skip auto-generated date tables and hidden tables
            if table.Name.startswith(('DateTableTemplate_', 'LocalDateTable_')) or table.IsHidden:
                continue
            
            all_user_tables.append(table.Name)
            
            # Column.IsDefaultLabel corresponds to Table.RowLabel TOM property
            # When using PBIP format, this is stored in TMDL and version-controlled
            row_label_column = next((col.Name for col in table.Columns if col.IsDefaultLabel and not col.IsHidden), None)
            
            if row_label_column:
                tables_with_row_labels.append((table.Name, row_label_column))
            else:
                tables_without.append(table.Name)

    print(f"📊 Total user tables analyzed: {len(all_user_tables)}")
    print()
    
    if tables_with_row_labels:
        print(f"✅ {len(tables_with_row_labels)} table(s) have row labels configured:")
        for tbl, col in tables_with_row_labels[:8]:
            print(f"     • {tbl} → [{col}]")
        if len(tables_with_row_labels) > 8:
            print(f"     ... and {len(tables_with_row_labels) - 8} more")
        print()

    if tables_without:
        print(f"⚠️  {len(tables_without)} table(s) do NOT have row labels set:")
        for tbl in tables_without[:10]:
            print(f"     • {tbl}")
        if len(tables_without) > 10:
            print(f"     ... and {len(tables_without) - 10} more")
        
        print("\n   📍 WHY THIS MATTERS:")
        print("      Row labels tell Copilot/Data Agent which column to display for each table.")
        print("      Without row labels, the AI may choose the wrong column (e.g., ID instead of Name).")
        print("\n   🎯 RECOMMENDATION:")
        print("      Set row labels on DIMENSION tables (Customer, Product, Store, Employee, etc.)")
        print("      For fact tables (Sales, Orders, Transactions), row labels are typically not needed.")
        print("\n   ⚙️  HOW TO SET:")
        print("      Power BI Desktop → Model view → select table → Advanced properties:")
        print("         1. Row label: Choose the column that describes each row (e.g., 'Customer Name')")
        print("         2. Key column: Choose the column that uniquely identifies each row (e.g., 'CustomerID')")
        print("\n   💡 TIP: Both Row Label AND Key Column should be set on dimension tables for best results.")
        print("\n   💾 Storage: Row labels are stored in TMDL (when using PBIP format)")
        print("      and are version-controlled via Git. Works for Import, DirectQuery, and Direct Lake.")
        
        issues.append(("important", "1.8 Row Labels", f"{len(tables_without)} tables without row labels"))
        score = max(1, 5 - min(len(tables_without) // 2, 4))
    else:
        print("✅ PASSED: All tables have row labels configured.")

    print("\n📚 Learn more:")
    print("   https://learn.microsoft.com/power-bi/natural-language/q-and-a-tooling-intro#set-a-row-label")
    print("   https://blog.crossjoin.co.uk/2024/08/25/tuning-power-bi-copilot-with-row-labels-and-key-columns/")

except Exception as e:
    print(f"ℹ️  Could not inspect row labels: {e}")
    print("\n   Manual check: Power BI Desktop → Model view → select table → Advanced → Row label")
    score = 3

print(f"\n📊 Score: {score}/5")
check_scores['1.8_row_labels'] = {'achieved': score, 'max': 5, 'weight': 2}
all_issues.extend(issues)

## 1.9 🔢 Explicit Measures (No Implicit Aggregations)

**Why it matters:** Numeric columns with default summarization create implicit measures that produce unpredictable results.

**What's checked:** Numeric columns with summarization ≠ "None"

In [ ]:
print("=" * 70)
print("CHECK 1.9: EXPLICIT MEASURES (NO IMPLICIT AGGREGATIONS)")
print("=" * 70)

issues = []
score = 15

NUMERIC_TYPES = ['Int64', 'Double', 'Decimal', 'Currency', 'Single',
                 'Whole number', 'Decimal number', 'Fixed decimal number']
SAFE_SUMMARIZE = {'None', 'DoNotSummarize', 'none', 'Do Not Summarize', 'donotsummarize'}

if 'Summarize By' not in columns_df.columns:
    print("ℹ️  'Summarize By' column not found — skipping check.")
    print("\n   Manual check: Power BI Desktop → Column tools → Summarization")
    score = 10
else:
    num_cols = visible_columns[visible_columns['Data Type'].isin(NUMERIC_TYPES)]
    implicit = num_cols[
        ~num_cols['Summarize By'].astype(str).isin(SAFE_SUMMARIZE) &
        num_cols['Summarize By'].notna()
    ]

    if implicit.empty:
        print("✅ PASSED: All numeric columns have Summarize By = None.")
    else:
        pct = len(implicit) / max(len(num_cols), 1) * 100
        score = max(0, int(15 - pct / 5))

        print(f"❌ {len(implicit)} numeric column(s) with implicit summarization:\n")
        for tbl, grp in implicit.groupby('Table Name'):
            print(f"   {tbl}:")
            for _, row in grp.iterrows():
                print(f"      • {row.get('Column Name')} (Type: {row.get('Data Type')}, Summarize: {row.get('Summarize By')})")

        print("\n🚨 CRITICAL: Set all numeric columns to 'Don't summarize'.")
        print("   Then create explicit DAX measures for calculations.")
        print("\n   📍 Power BI Desktop → Column tools → Summarization → Don't summarize")
        issues.append(("critical", "1.9 Implicit Measures", f"{len(implicit)} columns with implicit summarization"))

print(f"\n📊 Score: {score}/15")
check_scores['1.9_implicit_measures'] = {'achieved': score, 'max': 15, 'weight': 3}
all_issues.extend(issues)

## 1.10 📊 Report-Scoped Measures

**Why it matters:** Report-scoped measures are NOT accessible to Data Agent. Migrate them to the semantic model to make them available.

**What's checked:** Detection via TOM (if available)

In [ ]:
print("=" * 70)
print("CHECK 1.10: REPORT-SCOPED MEASURES")
print("=" * 70)

print("ℹ️  This check requires access to the .pbix file or TOM model.\n")
print("🔍 Manual verification:")
print("   1. Open your reports in Power BI Desktop")
print("   2. Check Data pane for measures with a 📊 icon (report-scoped)")
print("   3. Move these to the semantic model if Data Agent should use them")
print("\n💡 TIP: Report-scoped measures are NOT visible to Data Agent.")
print("   Create them in the model instead.")

# Partial credit for awareness
score = 5
check_scores['1.10_report_measures'] = {'achieved': score, 'max': 5, 'weight': 1}
print(f"\n📊 Score: {score}/5 (manual verification required)")

## 1.11 📅 Ambiguous Date Fields

**Why it matters:** Multiple date columns without clear guidance confuse the AI when users ask time-based questions.

**What's checked:**
- Count of visible date columns
- Whether date columns have descriptions
- Risk level based on number of date columns

**Remediation:** Use AI Instructions to specify default date field, hide non-primary date columns, or add clear descriptions.

In [ ]:
print("=" * 70)
print("CHECK 1.11: AMBIGUOUS DATE FIELDS & MARKED DATE TABLE")
print("=" * 70)

issues = []
score = 5

DATE_TYPES = ['DateTime', 'Date', 'DateTimeOffset']

date_cols = visible_columns[
    visible_columns['Data Type'].isin(DATE_TYPES) |
    visible_columns['Data Type'].astype(str).str.lower().str.contains('date', na=False)
]

def desc_status(row):
    d = row.get('Description', '')
    return "has description" if d and str(d).strip() else "NO description"

# ------------------------------------------------------------------
# MARKED DATE TABLE CHECK (per MS Learn copilot-semantic-models)
# Without a marked date table, Copilot may filter time questions by
# the wrong column (e.g., Customer[Birthday]) instead of the real
# date table. This produced a documented incorrect-result example.
# ------------------------------------------------------------------
marked_date_tables = []
try:
    from sempy_labs.tom import connect_semantic_model
    with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
        for table in tom.model.Tables:
            if table.Name.startswith(('DateTableTemplate_', 'LocalDateTable_')):
                continue
            # DataCategory == 'Time' on a date column → table is marked as date table
            for col in table.Columns:
                try:
                    if str(getattr(col, 'DataCategory', '') or '').lower() == 'time':
                        marked_date_tables.append((table.Name, col.Name))
                        break
                except Exception:
                    continue
    if marked_date_tables:
        print("✅ Marked date table detected:")
        for tbl, col in marked_date_tables:
            print(f"     • {tbl}[{col}]  (DataCategory = Time)")
    else:
        print("⚠️  NO marked date table detected.")
        print("   Copilot may filter time-based questions by the WRONG date column")
        print("   (e.g., Customer[Birthday] instead of your fact date).")
        print("   ACTION: In Power BI Desktop → Table tools → 'Mark as date table'")
        print("           Choose your dimension date column. (Sets DataCategory='Time')")
        issues.append(("important", "1.11 Date Fields", "No marked date table — Copilot may filter the wrong date column"))
        score = max(1, score - 2)
except Exception as e:
    print(f"ℹ️  Could not inspect marked date table via TOM: {e}")
    print("   Manual check: Power BI Desktop → Table tools → 'Mark as date table'")

print()

# ------------------------------------------------------------------
# AMBIGUOUS DATE FIELDS CHECK
# ------------------------------------------------------------------
if date_cols.empty:
    print("✅ OK: No visible date columns found.")
    print("   Date columns may be hidden (role-playing dimension design).")
elif len(date_cols) == 1:
    r = date_cols.iloc[0]
    print(f"✅ PASSED: Single visible date column:")
    print(f"   {r.get('Table Name')}[{r.get('Column Name')}] — {desc_status(r)}")
elif len(date_cols) <= 3:
    print(f"ℹ️  INFO: {len(date_cols)} date columns found (low ambiguity risk):")
    for _, row in date_cols.iterrows():
        print(f"   • {row.get('Table Name')}[{row.get('Column Name')}] — {desc_status(row)}")
    print("\n💡 Consider adding AI Instructions to specify default date for common questions.")
    score = min(score, 4)
else:
    print(f"⚠️  WARNING: {len(date_cols)} visible date column(s) — HIGH ambiguity risk!\n")
    for tbl, grp in date_cols.groupby('Table Name'):
        print(f"   Table: {tbl}")
        for _, row in grp.iterrows():
            print(f"      • {row.get('Column Name')} ({desc_status(row)})")

    score = max(0, 5 - max(0, len(date_cols) - 3))
    issues.append(("important", "1.11 Date Fields", f"{len(date_cols)} visible date columns without clear AI guidance"))

    print("\n📍 ACTION:")
    print("   1. Hide date columns users should not query directly")
    print("      (especially Customer[Birthday], Employee[HireDate], etc. —")
    print("       per MS Learn, these are common Copilot filter mistakes)")
    print("   2. Add descriptions clarifying each date column's purpose")
    print("   3. Mark your primary date table (Table tools → 'Mark as date table')")
    print("   4. Prep for AI → AI Instructions:")
    print("      Example: 'When the user asks about dates, use [Order Date] by default'")
    print("   5. Create Verified Answers for most common time-based questions")

print("\n📚 Reference:")
print("   https://learn.microsoft.com/power-bi/create-reports/copilot-semantic-models")

print(f"\n📊 Score: {score}/5")
check_scores['1.12_date_fields'] = {'achieved': score, 'max': 5, 'weight': 1}
all_issues.extend(issues)


## 1.12 👁️ Hidden Objects & Private Tables Risk (PK / FK / Sort / GUIDs)

**Why it matters (per [MS Learn — Copilot grounding](https://learn.microsoft.com/power-bi/create-reports/copilot-semantic-models)):**

Copilot **excludes** the following from its grounding context:
- Hidden columns and measures
- Hidden report pages
- **Tables marked as Private**

Copilot **includes** as grounding context:
- Visible tables/columns/measures
- Descriptions, data types, format strings, data category
- Full linguistic schema (synonyms / verbs)

**Implications:**
- Verified Answers break silently if they reference hidden columns
- **Technical columns** (primary keys, foreign keys, sort columns, GUIDs) MUST be hidden — Copilot never needs to see a relationship key or GUID to answer a business question
- For **internal-only tables** that should be entirely invisible to Copilot, mark them as **Private** (Power BI Desktop → Model view → Table → Properties → Private)

**What's checked:**
- Count of hidden objects
- Hidden columns lacking descriptions
- **Visible technical columns** (key/GUID/sort patterns) that should be hidden
- Verified Answers risk assessment

**Remediation:** Hide all primary keys, foreign keys, sort columns, and GUIDs. Mark internal-only tables as Private. Add descriptions to all remaining hidden columns referenced by Verified Answers.


In [ ]:
print("=" * 70)
print("CHECK 1.12: HIDDEN OBJECTS, PRIVATE TABLES & TECH COLUMNS")
print("=" * 70)

issues = []
score = 5

hidden_tables   = tables_df[is_hidden_mask(tables_df)]
hidden_columns  = columns_df[is_hidden_mask(columns_df)]
hidden_measures = measures_df[is_hidden_mask(measures_df)]

print(f"   Hidden objects summary:")
print(f"      Tables   : {len(hidden_tables):>4}")
print(f"      Columns  : {len(hidden_columns):>4}")
print(f"      Measures : {len(hidden_measures):>4}")
print()

# ------------------------------------------------------------------
# DETECT TABLES MARKED AS PRIVATE (excluded from Copilot grounding)
# Per MS Learn: tables marked as Private are excluded from Copilot's
# grounding context, even if they are not hidden.
# ------------------------------------------------------------------
private_tables = []
try:
    from sempy_labs.tom import connect_semantic_model
    with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
        for table in tom.model.Tables:
            if table.Name.startswith(('DateTableTemplate_', 'LocalDateTable_')):
                continue
            # ExcludeFromModelRefresh / IsPrivate vary by TOM version; check both
            is_private = False
            for attr in ('IsPrivate', 'ExcludeFromAutomaticAggregations'):
                try:
                    if bool(getattr(table, attr, False)):
                        is_private = True
                        break
                except Exception:
                    continue
            if is_private:
                private_tables.append(table.Name)
    if private_tables:
        print(f"ℹ️  {len(private_tables)} table(s) marked as Private (hidden from Copilot grounding):")
        for t in private_tables[:8]:
            print(f"      • {t}")
        print("   ✅ Good practice for internal-only tables you don't want Copilot to see.")
        print()
except Exception:
    pass  # TOM not available or attribute not exposed — silent skip

# ------------------------------------------------------------------
# DETECT VISIBLE TECHNICAL COLUMNS THAT SHOULD BE HIDDEN
# Per best practice: Copilot never needs to see a GUID or relationship
# key to answer a business question. Hide PKs, FKs, sort columns, GUIDs.
# ------------------------------------------------------------------
TECH_COL_PATTERN = (
    r'(^id$|_id$|^key$|_key$|^pk_|_pk$|^fk_|_fk$|guid|^sk_|_sk$|'
    r'sortorder|sort_order|sort_key|_sortby$|_index$|rowversion|etag|hashkey)'
)
visible_tech_cols = visible_columns[
    visible_columns['Column Name'].astype(str).str.lower().str.contains(TECH_COL_PATTERN, regex=True, na=False)
]

if not visible_tech_cols.empty:
    print(f"⚠️  {len(visible_tech_cols)} visible column(s) appear to be technical (PK/FK/sort/GUID):")
    for _, row in visible_tech_cols.head(15).iterrows():
        print(f"     • {row.get('Table Name')}[{row.get('Column Name')}]  (Type: {row.get('Data Type')})")
    if len(visible_tech_cols) > 15:
        print(f"     ... and {len(visible_tech_cols) - 15} more")
    print("\n   🚨 ACTION: Hide these columns. Copilot/Data Agent never needs to see")
    print("      a GUID or relationship key to answer a business question.")
    print("      Visible technical columns add noise and may be picked incorrectly")
    print("      during DAX generation.")
    issues.append(("important", "1.12 Hidden Objects",
                   f"{len(visible_tech_cols)} visible technical columns (PK/FK/sort/GUID) should be hidden"))
    score = max(1, score - 2)
else:
    print("✅ No visible technical (PK/FK/sort/GUID) columns detected.")

print()

# Hidden columns without descriptions — double risk
if not hidden_columns.empty:
    hc_no_desc = hidden_columns[
        hidden_columns['Description'].isna() | (hidden_columns['Description'].astype(str).str.strip() == '')
    ] if 'Description' in hidden_columns.columns else hidden_columns

    pct = len(hc_no_desc) / max(len(hidden_columns), 1) * 100

    if pct > 50:
        print(f"⚠️  WARNING: {len(hc_no_desc)} hidden column(s) lack descriptions ({pct:.0f}% of hidden columns).")
        print("   If any of these are referenced by Verified Answers, the answer will silently fail.")
        score = max(1, score - 2)
        issues.append(("important", "1.12 Hidden Objects", f"{len(hc_no_desc)} hidden columns without descriptions — Verified Answers risk"))
    else:
        print(f"✅ Most hidden columns have descriptions ({100-pct:.0f}% coverage).")
else:
    print("✅ No hidden columns found.")

print("\n🚨 REMINDER: Verified Answers will NOT work if they reference hidden columns.")
print("   Ensure all columns referenced in Verified Answers are VISIBLE in the model.")
print("\n💡 RULE OF THUMB (per MS Learn — Copilot grounding):")
print("   • Hide:    PKs, FKs, sort columns, GUIDs, audit timestamps, internal flags")
print("   • Private: Entire internal-only tables (excluded from Copilot grounding)")
print("   • Q&A:    Toggle OFF 'Include in Q&A' for fields you want to keep visible")
print("              in reports but exclude from Copilot/Q&A linguistic context.")
print("   • Visible: Business-meaningful names users would type in a question")

print("\n📚 Reference:")
print("   https://learn.microsoft.com/power-bi/create-reports/copilot-semantic-models")

print(f"\n📊 Score: {score}/5")
check_scores['1.13_hidden_objects'] = {'achieved': score, 'max': 5, 'weight': 1}
all_issues.extend(issues)


## 1.13 📦 Model Complexity / Bloat — Copilot Context Truncation Risk

**Why it matters:** Copilot and Data Agents serialize your semantic model schema (tables, columns, measures, calculated columns) into the LLM prompt as **grounding context**. Large models exceed the prompt budget and the schema gets **truncated before reaching the LLM** — Copilot literally never sees the omitted objects and will hallucinate, miss fields, or pick the wrong table without any visible error.

> 📘 **Official guidance:** [Prepare your data for AI: AI data schemas (Microsoft Learn)](https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai-data-schema)
>
> *"By using an AI data schema, semantic model authors can define a focused subset of the model's schema for Copilot to prioritize when it generates responses … A streamlined schema reduces ambiguity, which helps Copilot deliver clearer and more accurate responses."*

### What's checked (automated)
- Visible **tables** count (warn > 30, critical > 50)
- Visible **columns** count (warn > 300, critical > 500)
- Visible **measures** count (warn > 100, critical > 150)
- **Calculated columns** count (warn > 50, critical > 100 — recomputed at refresh, add schema noise)
- Total grounding object count (tables + columns + measures)
- Visible helper / intermediate measures (should be hidden)
- **Q&A enabled** prerequisite (Prep data for AI tabs are disabled if Q&A is off)

### Remediation — use *Prep data for AI → Simplify data schema*

Per Microsoft Learn, the official path is:

1. **Power BI Desktop → Home ribbon → Prep data for AI** (or in the service: semantic-model page ribbon)
2. **Turn on Q&A** for the model first — otherwise the Prep-for-AI tabs are disabled
3. Open the **Simplify data schema** tab and select **only** the fields Copilot should reason over — prioritize clean columns with limited ambiguity
4. Click **Apply** to save the AI data schema to the model

### Key nuances from the docs (do not skip)

- 🔸 **Hidden ≠ excluded forever.** Hidden fields are auto-excluded only on *initial* AI-data-schema setup. New columns added to the model later are **not** auto-excluded — re-open Prep for AI after every model change.
- 🔸 **Hierarchies have a gotcha.** If the AI data schema includes a hierarchy but a column of that hierarchy isn't individually selected on the table, the Copilot schema **still includes that column**. Pick fields individually if you need precise control.
- 🔸 **Relationships are always respected** regardless of the AI data schema — Copilot can traverse to related tables even if a related field isn't selected. So the schema scope must still align with the relationship graph.
- 🔸 **AI data schema does NOT apply to every Copilot feature.** Report summarization, "create a report page", "search for data", and DAX-query generation use the **entire** semantic model. Model-level cleanup (hiding tech columns, marking tables Private, removing dead calc columns) is still required.
- 🔸 **Each edit needs a Copilot pane refresh** (close & reopen) to take effect during testing.
- 🔸 **Consumers can't see or disable the AI data schema** — the author owns the trust boundary.

### Additional model-level actions (to keep total schema small)

- Mark internal-only tables as **Private** so Copilot ignores them entirely
- **Hide** technical columns (PKs, FKs, GUIDs, sort columns, audit timestamps)
- Push frequently-used **calculated columns into the source query / dataflow** (ETL)
- Hide **helper / intermediate measures** used only as building blocks
- Split very large models into **domain-specific Data Agents** (Sales, Finance, …) instead of one "everything" agent


In [ ]:
print("=" * 70)
print("CHECK 1.13: MODEL COMPLEXITY / BLOAT — COPILOT CONTEXT TRUNCATION RISK")
print("=" * 70)

issues = []
score = 5

# ------------------------------------------------------------------
# Identify calculated columns (DAX-defined columns added to the model)
# Different sempy versions expose this via 'Type', 'Column Type', or 'Kind'
# ------------------------------------------------------------------
calc_col_total = 0
calc_col_visible = 0
calc_col_type_field = next(
    (c for c in ['Type', 'Column Type', 'Kind'] if c in columns_df.columns),
    None
)
if calc_col_type_field:
    calc_mask_all = columns_df[calc_col_type_field].astype(str).str.lower().str.contains(
        'calculated', na=False
    )
    calc_col_total = int(calc_mask_all.sum())
    if not visible_columns.empty:
        calc_mask_vis = visible_columns[calc_col_type_field].astype(str).str.lower().str.contains(
            'calculated', na=False
        )
        calc_col_visible = int(calc_mask_vis.sum())
        visible_calc_cols = visible_columns[calc_mask_vis]
    else:
        visible_calc_cols = visible_columns.iloc[0:0]
else:
    visible_calc_cols = visible_columns.iloc[0:0]

vis_tbl_n = len(visible_tables)
vis_col_n = len(visible_columns)
vis_msr_n = len(visible_measures)
vis_calc_n = calc_col_visible
total_grounding_objs = vis_tbl_n + vis_col_n + vis_msr_n  # calc cols already counted in columns

# ------------------------------------------------------------------
# Q&A enabled prerequisite — Prep-for-AI tabs are disabled if Q&A is off
# (per https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai-data-schema)
# ------------------------------------------------------------------
qna_enabled = None
try:
    from sempy_labs.tom import connect_semantic_model
    with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
        # The model property is typically named 'DiscourageImplicitMeasures' or
        # 'DefaultPowerBIDataSourceVersion'; Q&A enablement lives under
        # model.QueryGroups / model.Model.DataAccessOptions or the
        # 'IsQNAFeatureEnabled' annotation on some versions. We probe defensively.
        for attr in ('IsQNAFeatureEnabled', 'QnaEnabled'):
            if hasattr(tom.model, attr):
                qna_enabled = bool(getattr(tom.model, attr))
                break
        if qna_enabled is None:
            # Fall back to annotation lookup
            try:
                for ann in tom.model.Annotations:
                    if 'qna' in ann.Name.lower() or 'questionsanswers' in ann.Name.lower():
                        qna_enabled = str(ann.Value).lower() in ('true', '1', 'enabled')
                        break
            except Exception:
                pass
except Exception:
    pass

# ------------------------------------------------------------------
# Display model size summary
# ------------------------------------------------------------------
print("   📐 Model size (objects sent to Copilot / LLM as grounding context):")
print(f"      Visible tables            : {vis_tbl_n:>5}")
print(f"      Visible columns           : {vis_col_n:>5}")
print(f"      Visible measures          : {vis_msr_n:>5}")
if calc_col_type_field:
    print(f"      Calculated columns (vis)  : {vis_calc_n:>5}  (total incl. hidden: {calc_col_total})")
else:
    print(f"      Calculated columns        :   n/a  (column type metadata not exposed by sempy)")
print(f"      ─────────────────────────────────")
print(f"      Total grounding objects   : {total_grounding_objs:>5}")
print()

# ------------------------------------------------------------------
# Thresholds — based on practical Copilot grounding limits
# (model schema is serialized into the LLM prompt; large models truncate)
# ------------------------------------------------------------------
THRESHOLDS = {
    'tables':      {'warn':  30, 'critical':  50},
    'columns':     {'warn': 300, 'critical': 500},
    'measures':    {'warn': 100, 'critical': 150},
    'calc_cols':   {'warn':  50, 'critical': 100},
    'total':       {'warn': 500, 'critical': 800},
}

def _bloat_check(label, n, key, unit_hint):
    """Emit warning/critical based on threshold and return score delta."""
    t = THRESHOLDS[key]
    if n > t['critical']:
        print(f"🔴 CRITICAL: {n} {label} — exceeds critical threshold ({t['critical']}).")
        print(f"   Copilot grounding will likely be TRUNCATED before reaching the LLM.")
        print(f"   {unit_hint}")
        issues.append(("critical", "1.13 Complexity",
                       f"{n} {label} exceeds critical threshold ({t['critical']}) — schema truncation risk"))
        return -2
    elif n > t['warn']:
        print(f"🟠 WARNING: {n} {label} — above recommended limit ({t['warn']}).")
        print(f"   Risk of Copilot context truncation; accuracy and latency will degrade.")
        print(f"   {unit_hint}")
        issues.append(("important", "1.13 Complexity",
                       f"{n} {label} exceeds warning threshold ({t['warn']})"))
        return -1
    return 0

PREP_HINT_TABLE = "→ Mark internal tables as Private; reduce scope via Prep data for AI → Simplify data schema."
PREP_HINT_COL   = "→ Hide PKs/FKs/GUIDs/sort cols; expose only business columns via Prep data for AI → Simplify data schema."
PREP_HINT_MSR   = "→ Hide helper/intermediate measures; include only KPI-level measures in the AI data schema."
PREP_HINT_CALC  = "→ Push calc-column logic into the source query / dataflow (ETL) instead of DAX."
PREP_HINT_TOTAL = "→ Split into domain-specific Data Agents (Sales / Finance / …) instead of one 'everything' agent."

score += _bloat_check("visible tables",   vis_tbl_n,  'tables',   PREP_HINT_TABLE)
score += _bloat_check("visible columns",  vis_col_n,  'columns',  PREP_HINT_COL)
score += _bloat_check("visible measures", vis_msr_n,  'measures', PREP_HINT_MSR)
if calc_col_type_field:
    score += _bloat_check("calculated columns", vis_calc_n, 'calc_cols', PREP_HINT_CALC)
score += _bloat_check("total grounding objects", total_grounding_objs, 'total', PREP_HINT_TOTAL)

# Show top tables by visible calculated-column count (helps prioritize remediation)
if calc_col_type_field and vis_calc_n > 0 and 'Table Name' in visible_calc_cols.columns:
    top_calc_tables = (
        visible_calc_cols.groupby('Table Name')['Column Name']
        .count()
        .sort_values(ascending=False)
        .head(5)
    )
    print("\n   Top tables with calculated columns:")
    for tbl, cnt in top_calc_tables.items():
        print(f"      • {tbl}: {cnt} calculated column(s)")

# ------------------------------------------------------------------
# Detect helper measures that are visible (they should be hidden)
# ------------------------------------------------------------------
HELPER_PATTERN = r'\b(helper|_h$|aux|auxiliary|temp|tmp|working|intermediate|_int$|__[a-z])\b'
visible_helpers = visible_measures[
    visible_measures['Measure Name'].astype(str).str.lower().str.contains(HELPER_PATTERN, regex=True, na=False)
]

if not visible_helpers.empty:
    print(f"\n⚠️  {len(visible_helpers)} potentially visible helper/intermediate measure(s):")
    for _, row in visible_helpers.head(10).iterrows():
        print(f"      • {row.get('Table Name')}[{row.get('Measure Name')}]")
    if len(visible_helpers) > 10:
        print(f"      ... and {len(visible_helpers) - 10} more")
    print("   → Hide them or exclude them from your AI data schema (Prep data for AI).")
    issues.append(("important", "1.13 Complexity",
                   f"{len(visible_helpers)} visible helper/intermediate measure(s)"))
    score -= 1

# ------------------------------------------------------------------
# Q&A enablement prerequisite for Prep-for-AI
# ------------------------------------------------------------------
print()
if qna_enabled is True:
    print("✅ Q&A is enabled — Prep data for AI tabs are accessible.")
elif qna_enabled is False:
    print("🔴 CRITICAL: Q&A is DISABLED on this model.")
    print("   The Prep data for AI → Simplify data schema tab will be disabled until")
    print("   you enable Q&A (Power BI Desktop → File → Options → Current file → Q&A).")
    issues.append(("critical", "1.13 Complexity",
                   "Q&A disabled — Prep for AI / AI data schema cannot be configured"))
    score -= 2
else:
    print("ℹ️  Could not determine Q&A enablement programmatically.")
    print("   Verify manually: Q&A must be ON for Prep data for AI tabs to be enabled.")

if not issues:
    print("\n✅ PASSED: Model complexity is within healthy limits for Copilot grounding.")

# ------------------------------------------------------------------
# Always-on guidance about truncation + Prep for AI
# ------------------------------------------------------------------
print()
print("📍 WHY THIS MATTERS FOR COPILOT / DATA AGENT:")
print("   The semantic model schema is serialized into the LLM prompt as grounding")
print("   context. Large models exceed the prompt budget and the schema gets")
print("   TRUNCATED — Copilot literally never sees the omitted tables/columns and")
print("   will hallucinate, miss fields, or pick the wrong table without any error.")
print()
print("✅ HOW TO FIX — Prep data for AI → Simplify data schema:")
print("   1. Power BI Desktop → Home ribbon → Prep data for AI")
print("      (or in the service: semantic-model page ribbon)")
print("   2. Ensure Q&A is ON (otherwise the Prep-for-AI tabs are disabled)")
print("   3. Open the 'Simplify data schema' tab")
print("   4. Select ONLY the fields Copilot should reason over — prioritize clean")
print("      columns with limited ambiguity; remove confusing/duplicate fields")
print("   5. Click Apply to save")
print()
print("⚠️  IMPORTANT NUANCES (per MS Learn):")
print("   • Hidden fields are auto-excluded only on FIRST setup — re-open Prep for AI")
print("     after model changes to re-evaluate newly added columns.")
print("   • Hierarchy gotcha: if a hierarchy is selected but its column isn't picked")
print("     individually, Copilot's schema STILL includes that column.")
print("   • Relationships are always respected — Copilot can traverse to related")
print("     tables even if related fields aren't in the AI data schema.")
print("   • AI data schema does NOT apply to: report summarization, 'create a report")
print("     page', 'search for data', or DAX-query generation — those use the FULL")
print("     model. Model-level cleanup (hiding tech columns, marking Private) is")
print("     still required.")
print("   • After editing, close & reopen the Copilot pane to refresh.")
print()
print("💡 ADDITIONAL MODEL-LEVEL ACTIONS:")
print("   • Mark internal-only tables as Private")
print("   • Hide PKs, FKs, GUIDs, sort columns, audit timestamps")
print("   • Push calc-column logic into source query / dataflow")
print("   • Split very large models into domain-specific Data Agents")
print()
print("📚 References:")
print("   • https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai-data-schema")
print("   • https://learn.microsoft.com/power-bi/create-reports/copilot-evaluate-data")
print("   • https://learn.microsoft.com/power-bi/create-reports/copilot-semantic-models")

score = max(0, score)
print(f"\n📊 Score: {score}/5")
check_scores['1.14_model_bloat'] = {'achieved': score, 'max': 5, 'weight': 1}
all_issues.extend(issues)


## 1.14 ⚡ Performance Analyzer for AI Data Schema Measures

**Why it matters:** Slow-executing measures in your AI Data Schema cause timeout errors and poor user experience. You must validate performance specifically for the measures selected in Prep for AI.

**What's checked:** Manual — run Performance Analyzer in Power BI Desktop targeting only measures included in AI Data Schema

**Key insights:**
- Measures exceeding 2 seconds warrant optimization before deployment
- Storage Engine vs Formula Engine query ratio
- Direct Lake fallback scenarios that trigger Import mode scans

In [ ]:
print("=" * 70)
print("CHECK 1.14: PERFORMANCE ANALYZER FOR AI DATA SCHEMA MEASURES")
print("=" * 70)

print("🔵 MANUAL CHECK — requires Power BI Desktop.\n")
print("📍 HOW TO RUN:")
print("   Power BI Desktop → View tab → Performance Analyzer → Start recording")
print("   Then create visuals using each measure in your AI Data Schema.\n")
print("✅ CHECKLIST:")
print()
print("   [ ] 1. Open report connected to this semantic model in Power BI Desktop")
print("   [ ] 2. Enable Performance Analyzer (View → Performance Analyzer)")
print("   [ ] 3. Click 'Start recording'")
print("   [ ] 4. For each measure in AI Data Schema, create a simple visual using it")
print("   [ ] 5. Review DAX query times — target < 2 seconds per measure")
print("   [ ] 6. Export results: click '...' → 'Export data'")
print()
print("⚠️  WARNING THRESHOLDS:")
print("   • < 500ms  ✅ Excellent — no action needed")
print("   • 500ms–2s ⚠️  Acceptable — monitor under production load")
print("   • 2s–5s    🟠 Slow — optimize before deployment")
print("   • > 5s     🔴 Critical — will cause Data Agent timeouts")
print()
print("💡 OPTIMIZATION ACTIONS:")
print("   • Run BPA (Check 1.2) to identify inefficient DAX patterns")
print("   • Remove column store scans with SUMMARIZECOLUMNS rewrites")
print("   • Use aggregation tables for large fact tables")
print("   • Review Direct Lake fallback if using Direct Lake mode (Check 1.3)")
print("   • Check if measure uses CALCULATE with complex filter contexts")
print()
print("📚 Documentation:")
print("   https://learn.microsoft.com/power-bi/create-reports/desktop-performance-analyzer")

score = 5 if prep_for_ai_configured else 2
check_scores['1.17_perf_analyzer'] = {'achieved': score, 'max': 5, 'weight': 1}
if not prep_for_ai_configured:
    all_issues.append(("important", "1.14 Perf Analyzer", "Run Performance Analyzer against AI Data Schema measures before deployment"))
print(f"\n📊 Score: {score}/5 (manual verification required)")

## 1.15 🔁 Duplicate Column Names Across Tables

**Why it matters:** When two tables expose a column with the same name (e.g., `Sales[Date]` and `Orders[Date]`), the Data Agent's DAX generator can pick the wrong table, leading to incorrect filter context and wrong results without any obvious error.

**What's checked:** Automated — scans all visible columns for names appearing in more than one table

In [ ]:
print("=" * 70)
print("CHECK 1.15: DUPLICATE COLUMN NAMES ACROSS TABLES")
print("=" * 70)

issues = []
score = 5

if 'Column Name' in visible_columns.columns and 'Table Name' in visible_columns.columns:
    # Find column names that appear in more than one table
    col_table_counts = (
        visible_columns
        .groupby('Column Name')['Table Name']
        .agg(list)
        .reset_index()
    )
    col_table_counts.columns = ['Column Name', 'Tables']
    col_table_counts['Table Count'] = col_table_counts['Tables'].apply(len)

    duplicates = col_table_counts[col_table_counts['Table Count'] > 1].sort_values('Table Count', ascending=False)

    if duplicates.empty:
        print("✅ PASSED: No duplicate column names found across tables.")
    else:
        print(f"⚠️  {len(duplicates)} column name(s) appear in multiple tables:\n")
        for _, row in duplicates.head(20).iterrows():
            tables_str = ', '.join(row['Tables'][:5])
            if len(row['Tables']) > 5:
                tables_str += f" ... +{len(row['Tables']) - 5} more"
            print(f"     • [{row['Column Name']}]  →  {row['Table Count']} tables: {tables_str}")

        if len(duplicates) > 20:
            print(f"\n     ... and {len(duplicates) - 20} more duplicate column names")

        print("\n📍 WHY THIS CAUSES ERRORS:")
        print("   When you ask 'Filter by Date', the AI doesn't know which table's")
        print("   [Date] column to use — it may guess wrong, silently returning bad data.")
        print()
        print("✅ REMEDIATION OPTIONS:")
        print("   Option A (Preferred): Rename ambiguous columns to be table-specific")
        print("     Example: Sales[Date] → Sales[Order Date]")
        print("     Example: Orders[Date] → Orders[Delivery Date]")
        print("   Option B: Add clear descriptions to distinguish identical column names")
        print("   Option C: Hide the non-primary instance in the AI Data Schema")
        print("   Option D: Add AI Instructions to specify which table to use by default")
        print()
        print("💡 FOCUS: Prioritize columns included in your AI Data Schema.")

        penalty = min(4, len(duplicates))
        score = max(1, 5 - penalty)
        issues.append(("important", "1.15 Duplicate Columns", f"{len(duplicates)} column name(s) shared across multiple tables"))
else:
    print("ℹ️  Column metadata not available — skipping check.")

score = max(0, score)
print(f"\n📊 Score: {score}/5")
check_scores['1.18_duplicate_cols'] = {'achieved': score, 'max': 5, 'weight': 1}
all_issues.extend(issues)

## 1.16 🧠 Copilot Grounding Data Completeness

**Why it matters:** Per [MS Learn — Use Copilot with semantic models](https://learn.microsoft.com/power-bi/create-reports/copilot-semantic-models), Copilot uses these model properties as **grounding context**:

- Descriptions (already covered in Check 1.6)
- **Format strings** — currency, %, decimals → tells Copilot how to present numbers
- **Data category** — Geography, Web URL, Image URL, Barcode, Time → tells Copilot which visual / interpretation to pick
- **Calculation group descriptions** — should list calculation item names (e.g., `YTD`, `MTD`, `PY`)
- Synonyms / linguistic schema (covered in Check 1.7)

**What's checked (automated):**
- Visible measures missing format strings
- Geography-like columns without a `Geography` data category set
- URL/Image-like columns without a `Web URL` / `Image URL` data category
- Calculation groups whose descriptions are empty or don't list calculation items


In [ ]:
print("=" * 70)
print("CHECK 1.16: COPILOT GROUNDING DATA COMPLETENESS")
print("=" * 70)

issues = []
score = 5

# ------------------------------------------------------------------
# 1) Format strings on visible measures
# ------------------------------------------------------------------
fmt_col = next((c for c in ['Format String', 'FormatString'] if c in measures_df.columns), None)
if fmt_col:
    msr_no_fmt = visible_measures[
        visible_measures[fmt_col].isna() | (visible_measures[fmt_col].astype(str).str.strip() == '')
    ]
    if not msr_no_fmt.empty:
        print(f"⚠️  {len(msr_no_fmt)} visible measure(s) missing a format string:")
        for _, r in msr_no_fmt.head(10).iterrows():
            print(f"     • {r.get('Table Name')}[{r.get('Measure Name')}]")
        if len(msr_no_fmt) > 10:
            print(f"     ... and {len(msr_no_fmt) - 10} more")
        print("   ACTION: Set currency / % / decimal format strings — Copilot uses them as grounding")
        print("           and will present numbers in the right format.")
        issues.append(("important", "1.16 Grounding",
                       f"{len(msr_no_fmt)} visible measure(s) without format strings"))
        score = max(1, score - 2)
    else:
        print("✅ All visible measures have format strings configured.")
else:
    print("ℹ️  Format string column not exposed by sempy — skipping format-string check.")

print()

# ------------------------------------------------------------------
# 2) Data category hints for Geography / URL / Image columns
# ------------------------------------------------------------------
cat_col = next((c for c in ['Data Category', 'DataCategory'] if c in columns_df.columns), None)

GEO_PATTERN = r'(country|state|province|region|city|postal|zip|continent|county|address|latitude|longitude)'
URL_PATTERN = r'(url|link|href|website|webpage)'
IMG_PATTERN = r'(image|photo|picture|thumbnail|imageurl|imgurl)'

geo_missing = url_missing = img_missing = pd.DataFrame()
if cat_col:
    is_text = visible_columns['Data Type'].astype(str).str.contains('string|text', case=False, na=False)
    no_cat = visible_columns[cat_col].isna() | \
             (visible_columns[cat_col].astype(str).str.strip().isin(['', 'Uncategorized', 'None']))

    geo_missing = visible_columns[
        is_text & no_cat &
        visible_columns['Column Name'].astype(str).str.lower().str.contains(GEO_PATTERN, na=False)
    ]
    url_missing = visible_columns[
        is_text & no_cat &
        visible_columns['Column Name'].astype(str).str.lower().str.contains(URL_PATTERN, na=False)
    ]
    img_missing = visible_columns[
        is_text & no_cat &
        visible_columns['Column Name'].astype(str).str.lower().str.contains(IMG_PATTERN, na=False)
    ]

    if not geo_missing.empty:
        print(f"⚠️  {len(geo_missing)} geography-like column(s) without a Data Category:")
        for _, r in geo_missing.head(8).iterrows():
            print(f"     • {r.get('Table Name')}[{r.get('Column Name')}]")
        print("   ACTION: Set Data Category to Country/Region, State, City, etc.")
        issues.append(("recommended", "1.16 Grounding",
                       f"{len(geo_missing)} geo-like column(s) without Data Category"))

    if not url_missing.empty:
        print(f"\n⚠️  {len(url_missing)} URL-like column(s) without a 'Web URL' Data Category:")
        for _, r in url_missing.head(8).iterrows():
            print(f"     • {r.get('Table Name')}[{r.get('Column Name')}]")
        issues.append(("recommended", "1.16 Grounding",
                       f"{len(url_missing)} URL-like column(s) without Web URL category"))

    if not img_missing.empty:
        print(f"\n⚠️  {len(img_missing)} image-like column(s) without an 'Image URL' Data Category:")
        for _, r in img_missing.head(8).iterrows():
            print(f"     • {r.get('Table Name')}[{r.get('Column Name')}]")
        issues.append(("recommended", "1.16 Grounding",
                       f"{len(img_missing)} image-like column(s) without Image URL category"))

    if geo_missing.empty and url_missing.empty and img_missing.empty:
        print("✅ Geography / URL / Image columns either categorized or absent.")
else:
    print("ℹ️  Data Category column not exposed by sempy — skipping category check.")

print()

# ------------------------------------------------------------------
# 3) Calculation group descriptions should list calculation item names
# ------------------------------------------------------------------
try:
    from sempy_labs.tom import connect_semantic_model
    cg_issues = []
    cg_total = 0
    with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
        for table in tom.model.Tables:
            try:
                cg = getattr(table, 'CalculationGroup', None)
            except Exception:
                cg = None
            if cg is None:
                continue
            cg_total += 1
            items = []
            try:
                items = [ci.Name for ci in cg.CalculationItems]
            except Exception:
                pass
            desc = (table.Description or '').strip() if hasattr(table, 'Description') else ''
            if not desc:
                cg_issues.append((table.Name, "no description"))
            else:
                # check at least 1 calc item name appears in description
                desc_lower = desc.lower()
                if items and not any(it.lower() in desc_lower for it in items):
                    cg_issues.append((table.Name, f"description doesn't list any of: {', '.join(items[:5])}"))

    if cg_total == 0:
        print("ℹ️  No calculation groups found.")
    else:
        if cg_issues:
            print(f"⚠️  {len(cg_issues)}/{cg_total} calculation group(s) need better descriptions:")
            for tbl, reason in cg_issues[:5]:
                print(f"     • {tbl} → {reason}")
            print("   ACTION: Per MS Learn, list calculation item names (YTD, MTD, PY, …)")
            print("           in the calculation group description so Copilot can use them.")
            issues.append(("important", "1.16 Grounding",
                           f"{len(cg_issues)} calculation group(s) without item names in description"))
            score = max(1, score - 1)
        else:
            print(f"✅ All {cg_total} calculation group(s) have descriptions listing item names.")
except Exception as e:
    print(f"ℹ️  Could not inspect calculation groups via TOM: {e}")

print(f"\n📊 Score: {score}/5")
print("\n📚 Reference:")
print("   https://learn.microsoft.com/power-bi/create-reports/copilot-semantic-models")

check_scores['1.16_grounding'] = {'achieved': score, 'max': 5, 'weight': 2}
all_issues.extend(issues)


## 1.17 🪜 Hierarchies (Logical Groupings)

**Why it matters:** Per [MS Learn — Optimize your semantic model for Copilot](https://learn.microsoft.com/power-bi/create-reports/copilot-evaluate-data) (Model structure), clear hierarchies — especially on dimension tables that support drilling down — help Copilot navigate levels and generate more accurate, drill-aware answers.

**What's checked (automated):**
- Enumerates all hierarchies (and their level counts) via TOM
- Flags drill-down candidate dimension tables (Date, Geography, Product, Organization, Customer, etc.) that have **no hierarchy** defined

**Examples:** `Date`: Year › Quarter › Month › Day · `Geography`: Country/Region › State › City


In [ ]:
print("=" * 70)
print("CHECK 1.17: HIERARCHIES (LOGICAL GROUPINGS)")
print("=" * 70)

issues = []
score = 5

# Per MS Learn (Optimize your semantic model for Copilot → Model structure):
# establish clear hierarchies, especially on dimension tables that support
# drilling down (e.g., Date: Year > Quarter > Month > Day; Geography:
# Country/Region > State > City). Hierarchies help Copilot navigate levels.
DRILL_PATTERN = r'(date|calendar|time|geography|geo|location|product|organi[sz]ation|org|customer|employee|account)'

try:
    from sempy_labs.tom import connect_semantic_model
    hierarchies = []            # (table, hierarchy, level_count)
    tables_with_hier = set()
    with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
        for table in tom.model.Tables:
            try:
                for h in table.Hierarchies:
                    lvl = 0
                    try:
                        lvl = len(list(h.Levels))
                    except Exception:
                        pass
                    hierarchies.append((table.Name, h.Name, lvl))
                    tables_with_hier.add(table.Name)
            except Exception:
                continue

    if hierarchies:
        print(f"✅ {len(hierarchies)} hierarchy(ies) found across {len(tables_with_hier)} table(s):\n")
        for tname, hname, lvl in hierarchies[:12]:
            print(f"     • {tname}[{hname}]  ({lvl} level(s))")
        if len(hierarchies) > 12:
            print(f"     ... and {len(hierarchies) - 12} more")
    else:
        print("⚠️  No hierarchies found in the model.")

    # Flag drill-down-friendly visible tables that have NO hierarchy
    candidate_tables = [
        t for t in visible_tables['Name'].dropna()
        if re.search(DRILL_PATTERN, str(t).lower())
    ]
    missing_hier = [t for t in candidate_tables if t not in tables_with_hier]

    if missing_hier:
        print(f"\n⚠️  {len(missing_hier)} drill-down candidate table(s) without a hierarchy:")
        for t in missing_hier[:10]:
            print(f"     • {t}")
        if len(missing_hier) > 10:
            print(f"     ... and {len(missing_hier) - 10} more")
        print("\n   ACTION: Add hierarchies (e.g., Date: Year > Quarter > Month > Day,")
        print("           Geography: Country/Region > State > City) so Copilot can drill down.")
        issues.append(("recommended", "1.17 Hierarchies",
                       f"{len(missing_hier)} drill-down table(s) without a hierarchy"))
        score = max(2, score - 2)
    elif hierarchies:
        print("\n✅ Key dimension tables have hierarchies configured.")

    if not hierarchies and not candidate_tables:
        print("\nℹ️  No obvious drill-down dimensions detected — hierarchies optional for this model.")

except Exception as e:
    print(f"ℹ️  Could not inspect hierarchies via TOM: {e}")
    print("   Manual check: Model view → dimension table → right-click column → New hierarchy")
    score = 3

print(f"\n📊 Score: {score}/5")
print("\n📚 Reference (Model structure):")
print("   https://learn.microsoft.com/power-bi/create-reports/copilot-evaluate-data")

check_scores['1.17_hierarchies'] = {'achieved': score, 'max': 5, 'weight': 1}
all_issues.extend(issues)


## 1.18 ✂️ Description Length — Copilot 200-Character Window

**Why it matters:** Per [MS Learn — Optimize your semantic model for Copilot](https://learn.microsoft.com/power-bi/create-reports/copilot-evaluate-data) (DAX query considerations), **Copilot uses only the first 200 characters** of any table, column, or measure description as grounding context. Anything beyond 200 characters is ignored.

**What's checked (automated):**
- Visible tables, columns, and measures whose descriptions exceed 200 characters
- Flags them so you can **front-load** the key business meaning and intended usage into the first 200 characters


In [ ]:
print("=" * 70)
print("CHECK 1.18: DESCRIPTION LENGTH — COPILOT 200-CHARACTER WINDOW")
print("=" * 70)

issues = []
score = 5

# Per MS Learn (Optimize your semantic model for Copilot → DAX query
# considerations): "Copilot uses only the first 200 characters" of a
# description. Front-load the key business meaning + intended usage so the
# most important grounding context fits inside the first 200 characters.
COPILOT_DESC_LIMIT = 200

def _over_limit(df, name_col):
    hits = []
    if 'Description' not in df.columns:
        return hits
    for _, row in df.iterrows():
        desc = str(row.get('Description') or '').strip()
        if len(desc) > COPILOT_DESC_LIMIT:
            if name_col == 'Name':
                label = str(row.get('Name', '?'))
            else:
                label = f"{row.get('Table Name', '?')}[{row.get(name_col, '?')}]"
            hits.append((label, len(desc)))
    return hits

long_tbls = _over_limit(visible_tables,   'Name')
long_cols = _over_limit(visible_columns,  'Column Name')
long_msrs = _over_limit(visible_measures, 'Measure Name')
total_long = len(long_tbls) + len(long_cols) + len(long_msrs)

if total_long == 0:
    print(f"✅ PASSED: All descriptions fit within the {COPILOT_DESC_LIMIT}-character Copilot grounding window.")
else:
    print(f"⚠️  {total_long} description(s) exceed {COPILOT_DESC_LIMIT} characters —")
    print(f"    Copilot only reads the FIRST {COPILOT_DESC_LIMIT} characters; the remainder is ignored.\n")
    for label, items in [("Tables", long_tbls), ("Columns", long_cols), ("Measures", long_msrs)]:
        if items:
            print(f"   {label} ({len(items)}):")
            for name, length in items[:8]:
                print(f"      • {name}  ({length} chars)")
            if len(items) > 8:
                print(f"      ... and {len(items) - 8} more")
    print("\n   ACTION: Front-load the business meaning + intended usage in the first")
    print(f"           {COPILOT_DESC_LIMIT} characters; move secondary detail to the end.")
    issues.append(("recommended", "1.18 Desc Length",
                   f"{total_long} description(s) exceed the {COPILOT_DESC_LIMIT}-char Copilot window"))
    score = max(2, 5 - min(3, total_long))

print(f"\n📊 Score: {score}/5")
print("\n📚 Reference (DAX query considerations):")
print("   https://learn.microsoft.com/power-bi/create-reports/copilot-evaluate-data")

check_scores['1.18_desc_length'] = {'achieved': score, 'max': 5, 'weight': 1}
all_issues.extend(issues)


---
# 2️⃣ PREP FOR AI — AI DATA SCHEMA

This section validates the **AI Data Schema** configuration (Prep for AI → Simplify data schema).

## 2.1 🎯 AI Data Schema Selection

**Why it matters:** This is THE most important configuration. Select ONLY tables/columns/measures relevant to Data Agent questions.

**What's checked:** Manual verification via checklist

In [ ]:
print("=" * 70)
print("CHECK 2.1: AI DATA SCHEMA SELECTION (MANUAL)")
print("=" * 70)

print("🚨 CRITICAL: This is a MANUAL configuration in Power BI.\n")
print("📍 Power BI Desktop or Service → Home → Prep data for AI → Simplify data schema")
print("\n✅ CHECKLIST:")
print("\n   [ ] 1. Define Data Agent scope (what questions should it answer?)")
print("   [ ] 2. Select ONLY relevant tables for those questions")
print("   [ ] 3. Include ALL dependent objects for selected measures (see Check 2.2)")
print("   [ ] 4. Exclude helper/intermediate measures")
print("   [ ] 5. Exclude duplicate measures (keep only canonical versions)")
print("   [ ] 6. Verify no hidden fields referenced by Verified Answers")
print("   [ ] 7. Ensure selected tables match Data Agent configuration (see Check 5)")
print("\n⚠️  IMPORTANT: Start narrow — it's easier to add objects than remove them.")
print("\n📚 Documentation:")
print("   https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai-data-schema")

score = 10 if prep_for_ai_configured else 0
check_scores['2.1_ai_data_schema'] = {'achieved': score, 'max': 20, 'weight': 3}
if not prep_for_ai_configured:
    all_issues.append(("critical", "2.1 AI Data Schema", "AI Data Schema not yet configured"))
print(f"\n📊 Score: {score}/20 (manual verification required)")

## 2.2 🔗 Measure Dependencies

**Why it matters:** If a measure depends on columns/measures not in AI Data Schema, DAX generation will fail or produce incorrect results.

**What's checked:** Measure dependencies via Semantic Link Labs

In [ ]:
print("=" * 70)
print("CHECK 2.2: MEASURE DEPENDENCIES ANALYSIS")
print("=" * 70)

try:
    from sempy_labs import get_measure_dependencies
    
    print("Retrieving measure dependencies...\n")
    deps = get_measure_dependencies(dataset=dataset, workspace=workspace)

    if deps is not None and not deps.empty:
        n_measures = deps['Measure Name'].nunique() if 'Measure Name' in deps.columns else len(deps)
        print(f"✅ Dependency data retrieved for {n_measures} measure(s).\n")
        print("📋 Sample dependencies (first 30 rows):")
        display(deps.head(30))
        
        print("\n🚨 CRITICAL ACTION:")
        print("   When configuring AI Data Schema, include ALL dependent objects for each")
        print("   measure you select. Missing dependencies cause incorrect DAX generation.")
        print("\n💡 WORKFLOW:")
        print("   1. Select measure in AI Data Schema")
        print("   2. Find its row in the dependency table above")
        print("   3. Add ALL referenced tables, columns, and measures to AI Data Schema")
        
        score = 15
        check_scores['2.2_dependencies'] = {'achieved': score, 'max': 15, 'weight': 3}
    else:
        print("ℹ️  No dependency data returned.")
        score = 10
        check_scores['2.2_dependencies'] = {'achieved': score, 'max': 15, 'weight': 3}

except ImportError:
    print("ℹ️  get_measure_dependencies not available.")
    print("\n   Install: %pip install semantic-link-labs --upgrade")
    score = 10
    check_scores['2.2_dependencies'] = {'achieved': score, 'max': 15, 'weight': 3}
except Exception as e:
    print(f"⚠️  Could not analyze dependencies: {e}")
    score = 10
    check_scores['2.2_dependencies'] = {'achieved': score, 'max': 15, 'weight': 3}

print(f"\n📊 Score: {score}/15")

## 2.3 🔧 Helper Measures Detection

**Why it matters:** Helper/intermediate measures visible in the model should be excluded from AI Data Schema to reduce noise.

**What's checked:** Visible measures with helper/intermediate naming patterns

In [ ]:
print("=" * 70)
print("CHECK 2.3: HELPER MEASURES DETECTION")
print("=" * 70)

issues = []
score = 5

HELPER_PATTERN = r'\b(helper|_h$|_helper$|aux|auxiliary|temp|tmp|working|intermediate|_int$|__[a-z]|_calc$|internal)\b'
visible_helpers = visible_measures[
    visible_measures['Measure Name'].astype(str).str.lower().str.contains(HELPER_PATTERN, regex=True, na=False)
]

if visible_helpers.empty:
    print("✅ PASSED: No obvious helper measures found in visible measures.")
else:
    print(f"⚠️  {len(visible_helpers)} potential helper/intermediate measure(s):\n")
    for _, row in visible_helpers.head(12).iterrows():
        print(f"     • {row.get('Table Name')}[{row.get('Measure Name')}]")
    if len(visible_helpers) > 12:
        print(f"     ... and {len(visible_helpers) - 12} more")
    
    print("\n📍 ACTION:")
    print("   1. Hide these measures if they're only building blocks")
    print("   2. Or exclude from AI Data Schema if they must stay visible")
    
    score = max(1, 5 - len(visible_helpers))
    issues.append(("important", "2.3 Helper Measures", f"{len(visible_helpers)} visible helper measures"))

print(f"\n📊 Score: {score}/5")
check_scores['2.3_helper_measures'] = {'achieved': score, 'max': 5, 'weight': 1}
all_issues.extend(issues)

---
# 3️⃣ PREP FOR AI — VERIFIED ANSWERS

**Critical configuration:** Create verified answers for your 5-10 most common or complex questions.

> 💡 **DevOps Insight:** Unlike Data Agent instructions (which live at service level with no Git story), Verified Answers travel with your model and benefit from full version control.

## 🔍 Where Verified Answers Live

- ✅ Are **fully honored by Data Agents** when present in the semantic model

**Verified Answers are stored in your semantic model's `linguisticMetadata` property** — the same location as AI Instructions and synonyms. This means they:

- ✅ Can be reviewed in pull requests
- ✅ Are version-controlled via TMDL when using PBIP format
- ✅ Flow through deployment pipelines with your model

In [ ]:
print("=" * 70)
print("CHECK 3: VERIFIED ANSWERS (MANUAL)")
print("=" * 70)

print("🚨 CRITICAL: Verified Answers stored in linguisticMetadata (version-controlled).\n")
print("📍 Power BI Desktop or Service → Home → Prep data for AI → Verified answers")
print("\n💾 STORAGE LOCATION:")
print("   • Verified Answers are stored in model's linguisticMetadata property")
print("   • When using PBIP format, they appear in TMDL files")
print("   • Fully version-controlled via Git when using source control")
print("   • Travel with the model through deployment pipelines")
print("\n✅ BEST PRACTICES CHECKLIST:")
print("\n   [ ] 1. Identify 5-10 most common questions from your team")
print("   [ ] 2. Create verified answer using appropriate visual type")
print("   [ ] 3. Add 5-7 complete, robust trigger questions (NOT partial phrases)")
print("   [ ] 4. Include both formal and conversational phrasings")
print("        Example triggers for 'YTD Revenue by Region':")
print("        - What is the year to date revenue by region?")
print("        - Show revenue YTD broken down by region")
print("        - YTD sales performance per region")
print("        - How are regions performing this year?")
print("   [ ] 5. Configure up to 3 filters for flexible slicing")
print("   [ ] 6. ⚠️  CRITICAL: Ensure ALL fields used are VISIBLE (not hidden)")
print("   [ ] 7. Test each trigger question for exact and semantic matching")
print("   [ ] 8. After renaming objects, re-save affected verified answers")
print("   [ ] 9. Review TMDL changes in pull requests (PBIP format)")
print("\n🚫 COMMON MISTAKES:")
print("   • Using partial phrases as triggers ('revenue by region')")
print("   • Referencing hidden columns → verified answer will FAIL silently")
print("   • Too few trigger questions (need 5-7 for good coverage)")
print("   • Not testing both formal and casual language")
print("   • Editing in service and losing version control")
print("\n✅ DEVOPS TIP:")
print("   Configure Verified Answers in Power BI Desktop, save as PBIP, commit to Git.")
print("   Changes to verified answers will appear in your TMDL diffs for review.")
print("\n📚 Documentation:")
print("   https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai-verified-answers")

score = 10 if prep_for_ai_configured else 0
check_scores['3_verified_answers'] = {'achieved': score, 'max': 15, 'weight': 3}
if not prep_for_ai_configured:
    all_issues.append(("critical", "3 Verified Answers", "Verified Answers not yet configured"))
print(f"\n📊 Score: {score}/15 (manual verification required)")

---
# 4️⃣ PREP FOR AI — AI INSTRUCTIONS

**Critical configuration:** Define business terminology, date defaults, and metric preferences.

> 🎯 **DevOps Insight:** When AI instructions live in the semantic model, they become part of PBIP folder structure, can be reviewed in pull requests, and flow through deployment pipelines. Data Agent instructions create configuration drift risk.

## 🔍 The Two Instruction Surfaces

**Push as much instruction content as possible into the semantic model layer** (Prep for AI), where it benefits from version control. Keep Data Agent instructions minimal — focused only on cross-source routing.

There are **two distinct places** where AI instructions can be configured:

## 💡 Best Practice Pattern

### 1️⃣ Semantic Model AI Instructions (Prep for AI)

**Storage:** `linguisticMetadata.CustomInstructions` property in model metadata

**Access:** Power BI Desktop → Prep data for AI → Add AI instructions

**Honored by:** Data Agent only

**Version Control:** ✅ Stored in TMDL when using PBIP format

**DevOps:** ✅ Diff-able in Git and flows through CI/CD pipelines

### 2️⃣ Data Agent Instructions (Fabric Service)

**Storage:** Service-level configuration (not in model metadata)

**Access:** Fabric Portal → Data Agent → Instructions

**Honored by:** Data Agents and Copilot in Power BI for that service configuration

**Version Control:** ❌ No TMDL representation, no Git story

**DevOps:** ⚠️ Manual tracking required, risk of configuration drift

In [ ]:
print("=" * 70)
print("CHECK 4: AI INSTRUCTIONS (MANUAL)")
print("=" * 70)

print("🚨 CRITICAL: AI Instructions at SEMANTIC MODEL level (version-controlled).\n")
print("📍 Power BI Desktop → Home → Prep data for AI → Add AI instructions")
print("\n💾 STORAGE & VERSION CONTROL:")
print("   • Instructions stored in linguisticMetadata.CustomInstructions")
print("   • linguisticMetadata version: 4.0.0 → 4.2.0 when instructions added")
print("   • Model annotation PBI_ProTooling includes 'CopilotTooling' flag")
print("   • When using PBIP format: fully version-controlled via TMDL")
print("   • Changes appear in Git diffs for review in pull requests")
print("\n📋 WHAT TO INCLUDE (SEMANTIC MODEL INSTRUCTIONS):")
print("\n   1️⃣  BUSINESS TERMINOLOGY & SYNONYMS (preferred over legacy Q&A Setup)")
print("      Define organization-specific terms and abbreviations in natural language:")
print("      Example: 'TMS is total media spend, calculated using [Total Media Spend]'")
print("      Example: 'VFR means Volume of Fruit Revenue'")
print("      Example: 'Top performer = sales rep achieving 110%+ of monthly quota'")
print("      💡 The AI Instructions pane is now the recommended place for synonyms")
print("         and obscure acronyms — superior to the legacy Q&A Setup dialog.")
print("\n   2️⃣  NEGATIVE SCOPE — TELL COPILOT WHAT'S NOT IN THE MODEL ⚠️ HIGH IMPACT")
print("      Explicitly state what data the model does NOT contain to prevent guessing.")
print("      Example: 'This model only contains data for the UK. If asked about US sales,")
print("                state that data is unavailable.'")
print("      Example: 'Data is limited to FY2023 onward. For prior years, no data exists.'")
print("      Example: 'Returns and refunds are not tracked in this model.'")
print("      Example: 'This model does not contain customer PII (no email, phone, address).'")
print("      → Prevents hallucinations and 'best guess' responses on out-of-scope questions.")
print("\n   3️⃣  TIME PERIOD DEFINITIONS")
print("      Clarify fiscal year, seasons, reporting periods:")
print("      Example: 'Fiscal year starts July 1'")
print("      Example: 'Peak season is November through January'")
print("\n   4️⃣  DEFAULT DATE FIELD")
print("      Specify which date to use when ambiguous:")
print("      Example: 'Unless specified, use Order Date for date-based questions'")
print("\n   5️⃣  METRIC PREFERENCES")
print("      Guide which measure to use for common questions:")
print("      Example: 'For profitability questions, use Contribution Margin'")
print("      Example: 'Revenue questions should use Net Revenue after discounts'")
print("\n   6️⃣  CALCULATION GROUPS & FIELD PARAMETERS")
print("      If using these features, explain how they work:")
print("      Example: 'Time intelligence is handled by the [Time Calc] calculation group'")
print("\n   7️⃣  COMPLEX DAX PATTERNS (optional)")
print("      Provide example DAX for complex scenarios to guide AI")
print("\n   8️⃣  DEFAULT GROUPINGS & ANALYSIS PREFERENCES")
print("      Define how the AI should group and analyze data by default:")
print("      Example: 'When analyzing sales performance, default grouping is Region then Product Category'")
print("      Example: 'Comparative analysis should default to prior year as the comparison period'")
print("      Example: 'Rank analyses should show Top 10 unless the user specifies otherwise'")
print("      Example: 'Currency values should be displayed in thousands (K) unless specified'")
print()
print("   [ ] Add Negative Scope statements (data the model does NOT contain)")
print("   [ ] Define cryptic acronyms / industry jargon (e.g., VFR, TMS)")
print("   [ ] Add default grouping preference to AI Instructions")
print("   [ ] Add default comparison period (e.g., prior year, prior month)")
print("   [ ] Specify default sort order for ranked questions")
print("   [ ] Define default aggregation level (product, category, region)")
print("\n🚫 DO NOT INCLUDE IN MODEL INSTRUCTIONS (use Data Agent instructions):")
print("   • Response formatting preferences")
print("   • Cross-data-source routing ('use lakehouse for X, semantic model for Y')")
print("   • Tone and presentation style")
print("\n⚠️  WHY THIS MATTERS — CONFIGURATION DRIFT RISK:")
print("   Your semantic model might be fully version-controlled with AI instructions")
print("   locked into your release pipeline. But Data Agent instructions sitting on")
print("   top have separate configuration someone edited last week — and nobody reviewed.")
print("\n🔄 DEVOPS PATTERN:")
print("   1. Configure instructions in Power BI Desktop (Prep for AI)")
print("   2. Save model as PBIP format")
print("   3. Commit TMDL files to Git")
print("   4. Deploy through pipeline (instructions travel with model)")
print("   5. Keep Data Agent instructions minimal (cross-source routing only)")

print("\n💡 TIP: Test responses BEFORE adding instructions to identify gaps.")
print("   Instructions should clarify, not contradict verified answers.")
print("\n📚 Documentation:")
print("   https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai-instructions")
print("   https://learn.microsoft.com/fabric/data-science/data-agent-semantic-model")

score = 10 if prep_for_ai_configured else 0
check_scores['4_ai_instructions'] = {'achieved': score, 'max': 15, 'weight': 3}
if not prep_for_ai_configured:
    all_issues.append(("critical", "4 AI Instructions", "AI Instructions not yet configured (incl. Negative Scope)"))
print(f"\n📊 Score: {score}/15 (manual verification required)")


## 4.1 🔍 Prep for AI Configuration Validator

**Why it matters:** Programmatically verify what's actually configured in Prep for AI (AI Data Schema, Custom Instructions, Verified Answers).

**What's checked:** Retrieves and displays actual Prep for AI configuration from the semantic model using Power BI MCP Server

> 💡 **Source:** Code adapted from [Fabric.guru by Sandeep Pawar](https://fabric.guru/programmatically-retrieve-prep-data-for-ai-configuration-of-semantic-models)

In [ ]:
print("=" * 70)
print("CHECK 4.1: PREP FOR AI CONFIGURATION VALIDATOR")
print("=" * 70)

print("📥 Retrieving Prep for AI configuration from semantic model...\n")

try:
    import httpx
    import json
    
    async def get_semantic_model_schema(model_identifier, workspace_name):
        """Fetch and parse the semantic model prep for ai config from Power BI MCP server.
        Author: Sandeep Pawar | Fabric.guru
        """
        # Resolve model ID from either dataset name or dataset ID.
        from sempy.fabric import list_datasets
        datasets = list_datasets(workspace=workspace_name)
        name_col = next((col for col in ['Dataset Name', 'Name'] if col in datasets.columns), None)
        id_col = next((col for col in ['Dataset ID', 'Id', 'Object ID'] if col in datasets.columns), None)
        
        if name_col is None or id_col is None:
            raise Exception(f"Could not identify dataset name/id columns. Available columns: {list(datasets.columns)}")

        model_str = str(model_identifier).strip()
        model_info = datasets[
            datasets[name_col].astype(str).str.strip().eq(model_str) |
            datasets[id_col].astype(str).str.strip().eq(model_str)
        ]

        if model_info.empty:
            return None
        
        model_id = str(model_info[id_col].iloc[0])
        
        MCP_SERVER_URL = "https://api.fabric.microsoft.com/v1/mcp/powerbi"
        token = notebookutils.credentials.getToken("pbi")
        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }

        payload = {
            "jsonrpc": "2.0",
            "id": 1,
            "method": "tools/call",
            "params": {
                "name": "GetSemanticModelSchema",
                "arguments": {"artifactId": model_id}
            }
        }

        def parse_sse_response(text):
            for line in text.split('\n'):
                if line.startswith('data: '):
                    return json.loads(line[6:])
            return {}

        async with httpx.AsyncClient(timeout=120.0) as client:
            response = await client.post(MCP_SERVER_URL, headers=headers, json=payload)
            data = parse_sse_response(response.text)

        parsed = json.loads(data['result']['content'][0]['text'])

        return {
            "name": parsed['semanticModel']['Name'],
            "tables": parsed['schema']['Tables'],
            "relationships": parsed['schema']['ActiveRelationships'],
            "custom_instructions": parsed['schema']['CustomInstructions'],
            "verified_answers": parsed['schema']['VerifiedAnswers']
        }

    # Retrieve configuration
    prep4ai = await get_semantic_model_schema(dataset, workspace)
    
    if prep4ai is None:
        print(f"❌ Could not retrieve Prep for AI configuration for model '{dataset}'")
        all_issues.append(("important", "4.1 Prep for AI Config", f"Could not retrieve Prep for AI configuration for dataset '{dataset}'"))
        score = 0
    else:
        print(f"✅ Successfully retrieved Prep for AI configuration for: {prep4ai['name']}\n")
        
        # AI Data Schema - Tables
        print("=" * 70)
        print("📊 AI DATA SCHEMA - TABLES")
        print("=" * 70)
        if prep4ai['tables']:
            print(f"\n✅ {len(prep4ai['tables'])} table(s) configured in AI Data Schema:\n")
            for table in prep4ai['tables']:
                table_name = table.get('Name', 'Unknown')
                columns = table.get('Columns', [])
                measures = table.get('Measures', [])
                print(f"   📋 {table_name}")
                print(f"      • Columns: {len(columns)}")
                print(f"      • Measures: {len(measures)}")
        else:
            print("\n⚠️  No tables found in AI Data Schema!")
            all_issues.append(("critical", "4.1 Prep for AI Config", "No tables found in AI Data Schema"))
        
        # Custom Instructions
        print("\n" + "=" * 70)
        print("📝 CUSTOM AI INSTRUCTIONS")
        print("=" * 70)
        if prep4ai['custom_instructions']:
            print(f"\n✅ Custom instructions configured:\n")
            print(prep4ai['custom_instructions'])
        else:
            print("\n⚠️  No custom AI instructions found.")
            all_issues.append(("important", "4.1 Prep for AI Config", "No custom AI instructions found in Prep for AI"))
        
        # Verified Answers
        print("\n" + "=" * 70)
        print("✔️  VERIFIED ANSWERS")
        print("=" * 70)
        if prep4ai['verified_answers']:
            print(f"\n✅ {len(prep4ai['verified_answers'])} verified answer(s) configured:\n")
            for idx, va in enumerate(prep4ai['verified_answers'], 1):
                print(f"   {idx}. Name: {va.get('Name', 'Unnamed')}")
                triggers = va.get('TriggerQuestions', [])
                print(f"      • Trigger questions: {len(triggers)}")
                if triggers:
                    print(f"      • Example: \"{triggers[0]}\"")
                
                # Show what fields are used (projections)
                projections = va.get('Projections', {})
                if projections:
                    tables_used = projections.get('Tables', [])
                    columns_used = projections.get('Columns', [])
                    measures_used = projections.get('Measures', [])
                    print(f"      • Uses: {len(tables_used)} table(s), {len(columns_used)} column(s), {len(measures_used)} measure(s)")
        else:
            print("\n⚠️  No verified answers found.")
            all_issues.append(("recommended", "4.1 Prep for AI Config", "No verified answers found in Prep for AI"))
        
        # Scoring
        score = 0
        if prep4ai['tables']:
            score += 5
        if prep4ai['custom_instructions']:
            score += 3
        if prep4ai['verified_answers']:
            score += 2
        
        print(f"\n💡 NOTE: This retrieves current configuration. You cannot modify Prep for AI programmatically.")
        print("   Use this to verify, monitor, or validate your configurations.\n")

except ImportError as e:
    print(f"⚠️  Required package not available: {e}")
    print("\n   Install: %pip install httpx --quiet")
    all_issues.append(("important", "4.1 Prep for AI Config", f"Required package missing: {e}"))
    score = 5
except Exception as e:
    print(f"⚠️  Could not retrieve Prep for AI configuration: {e}")
    print("\n   This check requires running in a Fabric notebook environment")
    print("   with access to notebookutils and the Power BI MCP server.")
    all_issues.append(("important", "4.1 Prep for AI Config", f"Prep for AI configuration retrieval failed: {e}"))
    score = 5

print(f"\n📊 Configuration Score: {score}/10")
check_scores['4.1_prep_config'] = {'achieved': score, 'max': 10, 'weight': 2}

print("\n📚 Learn more:")
print("   https://fabric.guru/programmatically-retrieve-prep-data-for-ai-configuration-of-semantic-models")
print("   https://learn.microsoft.com/power-bi/developer/mcp/remote-mcp-server-get-started")


---
# 5️⃣ DATA AGENT CONFIGURATION

**Critical requirement:** Select the SAME tables in Data Agent that you defined in Prep for AI → AI Data Schema.

In [ ]:
print("=" * 70)
print("CHECK 5: DATA AGENT CONFIGURATION (MANUAL)")
print("=" * 70)

print("🚨 CRITICAL: Data Agent table selection MUST match AI Data Schema.\n")
print("📍 Fabric Portal → Data Agent → Add semantic model")
print("\n✅ CONFIGURATION CHECKLIST:")
print("\n   [ ] 1. Add your prep'd semantic model to Data Agent")
print("   [ ] 2. ⚠️  Select THE SAME tables as in Prep for AI → AI Data Schema")
print("   [ ] 3. Mark the semantic model as 'Prepped for AI' in Power BI Service")
print("        → Signals to Standalone Copilot that this model should be prioritized")
print("        → Power BI Service → semantic model → Settings → Prepped for AI")
print("   [ ] 4. Test responses BEFORE adding Data Agent instructions")
print("   [ ] 5. Review DAX queries to verify accuracy")
print("   [ ] 6. (Power BI Desktop only) Use the 'Select Skills' dropdown when testing")
print("        → Force Copilot to test a specific capability (e.g., only 'Create Report Pages')")
print("        → Helps isolate issues during development")
print("\n🚫 DATA AGENT INSTRUCTIONS — DO NOT INCLUDE:")
print("   • Semantic model specific guidance (use Prep for AI → AI Instructions)")
print("   • Business terminology (use AI Instructions)")
print("   • Date field defaults (use AI Instructions)")
print("   • Measure preferences (use AI Instructions)")
print("   • Negative scope statements (use AI Instructions — see Section 4)")
print("\n✅ DATA AGENT INSTRUCTIONS — ONLY INCLUDE:")
print("   • Response formatting preferences")
print("   • Cross-source routing (for multiple data sources)")
print("   • Common abbreviations that apply across ALL sources")
print("   • Tone and presentation style")
print("\n💡 ROUTING EXAMPLE (multi-source agents):")
print("   'For revenue questions use the Sales semantic model'")
print("   'For real-time delivery status use the Operations KQL database'")
print("\n⚠️  REMEMBER: Data Agent instructions apply across ALL data sources.")
print("   Semantic model specific guidance MUST be in Prep for AI.")
print("\n📚 Documentation:")
print("   https://learn.microsoft.com/fabric/data-science/data-agent-semantic-model")

score = 0  # Must be configured separately
check_scores['5_data_agent_config'] = {'achieved': score, 'max': 20, 'weight': 3}
all_issues.append(("critical", "5 Data Agent Config", "Data Agent not yet configured — see checklist above"))
print(f"\n📊 Score: {score}/20 (manual verification required)")


---
# 6️⃣ TESTING & VALIDATION

**Critical phase:** Systematic testing before deployment ensures accuracy and identifies configuration gaps.

In [ ]:
print("=" * 70)
print("CHECK 6: TESTING & VALIDATION (MANUAL)")
print("=" * 70)

print("🚨 CRITICAL: Test systematically BEFORE adding AI Instructions.\n")
print("⏱️  IMPORTANT — DON'T EXPECT IMMEDIATE INDEXING:")
print("   After publishing a model, the text index (which helps Copilot understand")
print("   string values like 'West' or 'Apple') may take a few minutes to build.")
print("   If Copilot fails immediately after publish, WAIT ~5 MINUTES and try again.")
print()
print("✅ TESTING WORKFLOW:")
print("\n   1️⃣  BASELINE TESTING (before AI Instructions)")
print("      • Ask 10-20 common questions")
print("      • Review DAX query in each response")
print("      • Verify data accuracy")
print("      • Identify which questions fail or return incorrect results")
print("      → Use results to guide Prep for AI configuration tweaks")
print("\n   2️⃣  VERIFIED ANSWERS TESTING")
print("      • Test each trigger question individually")
print("      • Verify exact matches return the verified answer")
print("      • Test semantic variations")
print("      • Confirm filters work as expected")
print("\n   3️⃣  AI DATA SCHEMA VALIDATION")
print("      • Test questions using objects IN the AI Data Schema → should work")
print("      • Test questions using objects NOT in schema → should fail gracefully")
print("      → Confirms schema is correctly configured")
print("\n   4️⃣  ISOLATE CAPABILITIES (Power BI Desktop)")
print("      • Use the 'Select Skills' dropdown to force Copilot to test ONE capability")
print("        at a time (e.g., only 'Create Report Pages', only 'Summarize').")
print("      • Helps pinpoint whether failures are model, instruction, or skill issues.")
print("\n   5️⃣  USE DAX QUERY VIEW FOR DIAGNOSIS")
print("      • Open DAX Query View in Power BI Desktop and use Copilot there to:")
print("        - Generate test queries against your AI Data Schema measures")
print("        - EXPLAIN complex DAX (e.g., 'Explain how KEEPFILTERS works in this measure')")
print("      • Great for validating measure logic before exposing to Data Agent.")
print("\n   6️⃣  PERFORMANCE TESTING")
print("      • Measure response times for complex queries")
print("      • If slow: analyze DAX performance, optimize model, simplify instructions")
print("      • Use Performance Analyzer in Power BI Desktop")
print("\n   7️⃣  AUTOMATED EVALUATION (Python SDK)")
print("      • Create ground truth Q&A dataset")
print("      • Run batch evaluation")
print("      • Track accuracy metrics over time")
print("\n      📚 SDK: https://learn.microsoft.com/fabric/data-science/fabric-data-agent-sdk")
print("      📚 Evaluation: https://learn.microsoft.com/fabric/data-science/evaluate-data-agent")
print("\n   8️⃣  DIAGNOSTICS REVIEW")
print("      • Download diagnostics logs from failed/incorrect responses")
print("      • Identify which stage caused the issue:")
print("        - Intent recognition")
print("        - DAX generation")
print("        - Data retrieval")
print("      → Guides which configuration needs adjustment")
print("\n      📚 Diagnostics: https://learn.microsoft.com/fabric/data-science/evaluate-data-agent#diagnostics-button")
print("\n   9️⃣  ITERATION")
print("      • Based on testing results, adjust:")
print("        - AI Data Schema (add/remove objects)")
print("        - AI Instructions (clarify terminology, add Negative Scope)")
print("        - Verified Answers (add more triggers)")
print("      • Re-test after each change (allow ~5 min for re-indexing)")
print("\n💡 DEPLOYMENT:")
print("   [ ] Use Git integration for version control")
print("   [ ] Use Deployment Pipelines for dev → test → prod")
print("   [ ] Add Data Agent description before publishing")
print("   [ ] For M365 Copilot: add publishing instructions")
print("\n      📚 Git: https://learn.microsoft.com/fabric/data-science/data-agent-git-integration")
print("      📚 M365: https://learn.microsoft.com/fabric/data-science/data-agent-microsoft-365-copilot")

score = 0  # Must be done separately
check_scores['6_testing'] = {'achieved': score, 'max': 20, 'weight': 3}
all_issues.append(("critical", "6 Testing", "Systematic testing required before deployment"))
print(f"\n📊 Score: {score}/20 (manual validation required)")


---
# 7️⃣ SERVICE & ADMIN BEST PRACTICES

**Tenant- and capacity-level configuration that affects every Data Agent and Copilot user.**

These items are typically configured by Power BI / Fabric admins, not model authors. They round out the readiness picture.

## 7.1 ✅ Mark Semantic Model as "Prepped for AI"
- Power BI Service → semantic model → **Settings → Prepped for AI**
- Signals to **Standalone Copilot** (Power BI Home page) that this model should be **prioritized in search results**
- Without this flag, your model competes with every other model in the tenant

## 7.2 🔍 Standalone Copilot — "Needle in a Haystack" Search
- Standalone Copilot (Power BI Home page) can find specific reports or visuals across the entire tenant
- Example query: *"Find the funnel chart about sales"*
- Only works if content is **well-indexed** and models are marked **Prepped for AI**
- Encourage users to use it instead of manually browsing workspaces

## 7.3 ⚙️ Copilot Capacities (Admin)
- **Problem:** "Noisy neighbors" — heavy AI usage can slow down ETL and reporting pipelines on a shared capacity
- **Solution:** Admins can designate a **dedicated Fabric capacity solely for Copilot usage**
- Mission-critical pipelines stay isolated from interactive AI workloads
- Configure: Fabric Admin Portal → Capacity settings → Copilot

## 7.4 💎 F64 Capacity Requirement (per [MS Learn](https://learn.microsoft.com/power-bi/create-reports/copilot-semantic-models))
- Copilot in Power BI Desktop requires the desktop client to be **configured to consume Copilot from an F64+ workspace**
- This requirement applies **regardless of which semantic model** you connect to (Pro / PPU / F64)
- Configure: Power BI Desktop → File → Options → Preview features → Copilot → choose F64 workspace

## 7.5 🏷️ Endorsement & Tags as a Copilot Readiness Gate
- Per MS Learn, treat **Copilot readiness as a criteria for endorsement** (Promoted / Certified)
- Use **Fabric tags** to label semantic models as ready for Copilot consumption
- Helps data consumers identify which models they can confidently query with Copilot
- Configure: Workspace → semantic model → Endorsement / Tags

## 7.6 🧪 Use a Dedicated Testing Copy
- If your production model is messy, **create a copy** specifically for testing Copilot features
- Refine the copy first, then port refinements back to production
- Avoids destabilizing existing reports while you iterate

## 7.7 📐 Confirm Q&A Is Enabled (linguistic schema prerequisite)
- Power BI Desktop → File → Options → Current file → **Data Load** → *"Turn on Q&A to ask natural language questions about your data"*
- Required to add a linguistic schema (synonyms, verbs, "Include in Q&A" toggle)
- Copilot reuses the Q&A linguistic schema as grounding context


In [ ]:
print("=" * 70)
print("CHECK 7: SERVICE & ADMIN BEST PRACTICES (MANUAL)")
print("=" * 70)

print("✅ TENANT / CAPACITY / SERVICE CHECKLIST:\n")
print("   [ ] 7.1 Mark semantic model as 'Prepped for AI'")
print("           Power BI Service → semantic model → Settings → Prepped for AI")
print("           → Boosts visibility in Standalone Copilot search results.\n")
print("   [ ] 7.2 Educate users on Standalone Copilot (Power BI Home page)")
print("           → Use natural language to find reports/visuals tenant-wide")
print("           → Example: 'Find the funnel chart about sales'\n")
print("   [ ] 7.3 (Admin) Designate a dedicated Copilot capacity")
print("           Fabric Admin Portal → Capacity settings → Copilot")
print("           → Prevents 'noisy neighbor' impact on ETL / reporting pipelines.\n")
print("   [ ] 7.4 Confirm F64+ capacity is configured for Copilot in Power BI Desktop")
print("           Power BI Desktop → File → Options → Copilot → choose F64 workspace")
print("           → Required regardless of which semantic model you connect to.\n")
print("   [ ] 7.5 Use Endorsement (Promoted / Certified) and Fabric Tags as a")
print("           Copilot readiness gate so consumers know which models to trust.\n")
print("   [ ] 7.6 Use a dedicated TESTING COPY of production models")
print("           → Refine the copy first, then port refinements back to production.\n")
print("   [ ] 7.7 Confirm Q&A is enabled (linguistic schema prerequisite)")
print("           Power BI Desktop → Options → Current file → Data Load →")
print("           'Turn on Q&A to ask natural language questions about your data'\n")
print("   [ ] 7.8 Confirm tenant-level Copilot settings allow your workload")
print("           Fabric Admin Portal → Tenant settings → Copilot and Azure OpenAI Service\n")

print("⏱️  REMINDER: After publishing changes, allow ~5 minutes for the text index")
print("    to rebuild before testing Copilot — early failures are often just timing.\n")

print("📚 Documentation:")
print("   https://learn.microsoft.com/power-bi/create-reports/copilot-semantic-models")
print("   https://learn.microsoft.com/fabric/admin/copilot-fabric-admin-settings")
print("   https://learn.microsoft.com/power-bi/create-reports/copilot-introduction")
print("   https://learn.microsoft.com/fabric/governance/endorsement-overview")

score = 5 if prep_for_ai_configured else 0
check_scores['7_service_admin'] = {'achieved': score, 'max': 10, 'weight': 1}
if not prep_for_ai_configured:
    all_issues.append(("recommended", "7 Service & Admin",
                       "Service/admin best practices not yet validated (Prepped for AI, F64 capacity, Endorsement, Q&A enabled, etc.)"))
print(f"\n📊 Score: {score}/10 (manual verification required)")


---
# 🏆 FINAL READINESS SCORECARD

In [ ]:
print()
print("=" * 75)
print("   🤖  SEMANTIC MODEL DATA AGENT READINESS SCORECARD v2.2.1")
print(f"       Model: {dataset}  |  Workspace: {workspace}")
print("=" * 75)
print()

# Calculate weighted score
total_weighted = 0
total_max_weighted = 0

for key, data in check_scores.items():
    achieved = data['achieved']
    max_pts = data['max']
    weight = data['weight']
    total_weighted += achieved * weight
    total_max_weighted += max_pts * weight

overall_pct = (total_weighted / total_max_weighted * 100) if total_max_weighted > 0 else 0

# Check labels
CHECK_LABELS = {
    '0_scoping': ('0 Scoping & Design Assessment', 10, 2),
    '1.1_star_schema': ('1.1 Star Schema Validation', 20, 3),
    '1.2_bpa': ('1.2 Best Practice Analyzer', 15, 3),
    '1.3_memory': ('1.3 Memory Analyzer', 5, 1),
    '1.4_data_types': ('1.4 Data Type Validation', 10, 2),
    '1.5_naming': ('1.5 Business-Friendly Naming', 15, 3),
    '1.6_descriptions': ('1.6 Descriptions Coverage', 20, 3),
    '1.7_synonyms': ('1.7 Synonyms', 5, 1),
    '1.8_row_labels': ('1.8 Row Labels', 5, 2),
    '1.9_implicit_measures': ('1.9 Explicit Measures', 15, 3),
    '1.10_report_measures': ('1.10 Report-Scoped Measures', 5, 1),
    '1.12_date_fields': ('1.11 Date Fields & Marked Date Table', 5, 1),
    '1.13_hidden_objects': ('1.12 Hidden Objects + Private Tables', 5, 1),
    '1.14_model_bloat': ('1.13 Model Complexity / Bloat', 5, 1),
    '1.17_perf_analyzer': ('1.14 Perf Analyzer (AI Schema)', 5, 1),
    '1.18_duplicate_cols': ('1.15 Duplicate Column Names', 5, 1),
    '1.16_grounding': ('1.16 Copilot Grounding Completeness', 5, 2),
    '2.1_ai_data_schema': ('2.1 AI Data Schema', 20, 3),
    '2.2_dependencies': ('2.2 Measure Dependencies', 15, 3),
    '2.3_helper_measures': ('2.3 Helper Measures', 5, 1),
    '3_verified_answers': ('3 Verified Answers', 15, 3),
    '4_ai_instructions': ('4 AI Instructions (incl. Negative Scope)', 15, 3),
    '4.1_prep_config': ('4.1 Prep for AI Config Validator', 10, 2),
    '5_data_agent_config': ('5 Data Agent Configuration', 20, 3),
    '6_testing': ('6 Testing & Validation', 20, 3),
    '7_service_admin': ('7 Service & Admin Best Practices', 10, 1),
}

print(f"  {'Check':<45} {'Score':>12}   {'Progress'}")
print("  " + "─" * 73)

for key, (label, max_pts, weight) in CHECK_LABELS.items():
    if key in check_scores:
        achieved = check_scores[key]['achieved']
        pct = achieved / max_pts * 100 if max_pts > 0 else 0
        bar_fill = int(pct / 5)
        bar = "█" * bar_fill + "░" * (20 - bar_fill)
        
        if pct >= 80:
            icon = "✅"
        elif pct >= 50:
            icon = "⚠️ "
        else:
            icon = "❌"
        
        print(f"  {icon} {label:<44} {achieved:>3}/{max_pts:<3}  {bar}")

print("  " + "─" * 73)
print(f"  {'TOTAL SCORE (weighted)':<45} {int(overall_pct):>3}%")
print()

# Rating
if   overall_pct >= 90: rating, comment = "🟢 READY FOR DEPLOYMENT", "Excellent — monitor and iterate based on user feedback"
elif overall_pct >= 75: rating, comment = "🟡 MOSTLY READY", "Good foundation — address flagged items before deployment"
elif overall_pct >= 50: rating, comment = "🟠 NEEDS WORK", "Significant improvements needed before production use"
else:                   rating, comment = "🔴 NOT READY", "Complete critical configuration before deploying"

print(f"  Rating : {rating}")
print(f"  Status : {comment}")
print()

# Prioritized issues
critical = [(s, c, d) for s, c, d in all_issues if s == 'critical']
important = [(s, c, d) for s, c, d in all_issues if s == 'important']
recommended = [(s, c, d) for s, c, d in all_issues if s == 'recommended']
warning = [(s, c, d) for s, c, d in all_issues if s == 'warning']

if critical:
    print("  🔴 CRITICAL — Must fix before deployment:")
    for _, check, desc in critical:
        print(f"     • [{check}] {desc}")
    print()

if important:
    print("  🟠 IMPORTANT — Fix before production use:")
    for _, check, desc in important:
        print(f"     • [{check}] {desc}")
    print()

if recommended:
    print("  🟡 RECOMMENDED — Improves accuracy:")
    for _, check, desc in recommended:
        print(f"     • [{check}] {desc}")
    print()

if warning:
    print("  🟡 WARNING — Review before release:")
    for _, check, desc in warning:
        print(f"     • [{check}] {desc}")
    print()

if not (critical or important or recommended or warning):
    print("  ✅ No automated issues detected — complete manual checks above!")

print()
print("=" * 75)
print("  📚 KEY RESOURCES")
print("=" * 75)
print("  Official Checklist:")
print("    https://github.com/microsoft/fabric-toolbox/blob/main/samples/")
print("    data_agent_checklist_notebooks/Semantic%20Model%20Data%20Agent%20Checklist.md")
print("  Prep for AI Documentation:")
print("    https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai")
print("  Data Agent with Semantic Models:")
print("    https://learn.microsoft.com/fabric/data-science/data-agent-semantic-model")
print("  Evaluate Your Data Agent:")
print("    https://learn.microsoft.com/fabric/data-science/evaluate-data-agent")
print("  Python SDK:")
print("    https://learn.microsoft.com/fabric/data-science/fabric-data-agent-sdk")
print("  Semantic Link Labs:")
print("    https://github.com/microsoft/semantic-link-labs")
print("=" * 75)
print()

print("✅ Analysis complete! Review results and complete manual checks above.")
print("💾 Save this notebook for future reference and re-run after making changes.")


---
### 📝 Notes

**v2.4 Changes (Microsoft Learn — *Use Copilot with semantic models* alignment):**
- 🆕 Check 1.11: Marked Date Table detection (prevents Copilot filtering by `Birthday`)
- 🆕 Check 1.12: Tables marked as Private detection + `Include in Q&A` toggle guidance
- 🆕 Check 1.16 (NEW): Copilot grounding data completeness — format strings on measures, data category on geo/URL/image columns, calculation group descriptions list calculation item names
- 🆕 Section 7: F64 capacity requirement, Endorsement/Tags as readiness gate, Q&A enabled prerequisite
- 🆕 Synonyms (1.7): Q&A must be enabled, `Include in Q&A` toggle to exclude technical fields
- 🆕 Coverage matrix updated to reflect MS Learn grounding rules (INCLUDED vs EXCLUDED context)

**v2.3 Changes (Power BI Copilot Best Practices alignment):**
- 🆕 Section 7: Service & Admin Best Practices (Mark as Prepped for AI, Copilot Capacities, Standalone Copilot, Testing Copy)
- 🆕 Check 1.7: Over-synonymization detection + auto-generated synonym warning + AI Instructions guidance (vs legacy Q&A Setup)
- 🆕 Check 1.12: Visible technical column detection (PKs, FKs, sort columns, GUIDs)
- 🆕 Check 4: Negative Scope guidance (tell Copilot what data the model does *not* contain)
- 🆕 Check 5: Mark as 'Prepped for AI' + 'Select Skills' dropdown for isolated testing
- 🆕 Check 6: 5-minute indexing wait reminder + DAX Query View tip for measure explanations
- 🆕 Core Principle callout: *"AI is only as good as the data behind it."*

**v2.2 Changes:**
- 🆕 Check 1.14: Performance Analyzer for AI Data Schema measures (manual)
- 🆕 Check 1.15: Duplicate column names across tables ✅ Automated
- 🆕 Check 4.1: Prep for AI Configuration Validator ✅ Automated
- 🆕 Security requirements added to the Section 0 scoping checklist
- 🆕 Default groupings and analysis preferences added to the Section 4 AI Instructions checklist

**v2.1 Changes:**
- Added Data Agent configuration validation
- Enhanced testing & validation framework
- Row labels check for dimension tables
- Direct Lake optimization guidance
- Report-scoped measures detection
- Data type validation
- Calculation Groups awareness
- Weighted scoring system
- Aligned with official Microsoft checklist

**Manual Checks Required:**
- Memory Analyzer (run separately)
- Performance Analyzer for AI Data Schema measures (1.14)
- AI Data Schema configuration
- Verified Answers creation
- AI Instructions configuration (incl. Negative Scope, default groupings)
- Security requirements definition
- Data Agent setup (incl. Mark as Prepped for AI)
- Testing & validation workflow (allow ~5 min for re-indexing)
- Service & admin best practices (Copilot Capacity, F64 capacity, tenant settings, endorsement)

**Automated Checks:**
- Star schema validation ✅
- Best Practice Analyzer ✅
- Data type validation ✅
- Business-friendly naming ✅
- Descriptions (coverage & quality) ✅
- Synonyms (incl. over-synonymization & auto-gen warnings) ✅
- Row labels ✅
- Explicit measures / implicit aggregation check ✅
- Marked date table detection ✅
- Hidden / private tables + technical columns (PK/FK/sort/GUIDs) ✅
- Duplicate column names across tables ✅
- Helper measures ✅
- Measure dependencies ✅
- Copilot grounding completeness (format strings / data category / calc-group descriptions) ✅
- Prep for AI Configuration Validator ✅

**Key reference:**
- [Use Copilot with semantic models (MS Learn)](https://learn.microsoft.com/power-bi/create-reports/copilot-semantic-models)
- [Update your data model to work well with Copilot for Power BI (MS Learn)](https://learn.microsoft.com/power-bi/create-reports/copilot-evaluate-data)

---
*This notebook does not modify your semantic model. Run it periodically to validate readiness.*
